<a href="https://colab.research.google.com/github/farisahmedbhatti123/2D_PINN_Demo/blob/main/Crabtree_Loss_Sensitivity_SHAP_AutoDownload_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crabtree Creek — TCN vs LSTM: loss-function sensitivity and feature importance

**Integrated from your three fixed-2025 scripts**, with the original composite training objective preserved. No separate poster-visuals script or external Python files are required.

**Research design:** 25 causal predictors; 36 × 5-minute input steps (nominal 180-minute context); seven ΔH forecast horizons (15–180 min); train **2020–2023**, validation **2024**, test **2025**. Both architectures use the same chronological split, batch size, AdamW settings, storm-weighted sampler, seed, and early-stopping schedule. Inputs are standardized on training data only. Outputs remain stage change (ft). The benchmark is persistence (`ΔH=0`).

### Eight training arms

Each architecture (TCN and LSTM) is fitted independently with four objectives:

- **composite (original):** 0.5 × bounded MASE-based magnitude loss + 0.5 × smooth direction surrogate.
- **RMSE:** square root of mean squared ΔH residual.
- **NSE:** horizon-wise normalized squared error using fixed **training-period stage variances**. This is an NSE-inspired, differentiable loss with stable denominators. The final reported **NSE** is conventional `1 − SSE_test/SST_test` on 2025 observed *absolute stages*.
- **KGE:** regularized differentiable 2009 KGE with correlation, standard-deviation ratio (α), and mean ratio (β), computed on reconstructed *absolute stage* per horizon. The final reported **KGE** uses the ordinary unregularized 2009 formula and can be undefined for constant series.

**Key statistical distinction:** Full-test NSE and KGE are **evaluation metrics**, not the exact minibatch scalar used for backpropagation. Training on NSE and RMSE can yield similar results because they share squared-error numerators, though NSE weights horizons by training variance.

### Outputs

The notebook writes eight model manifests and predictions, all-horizon loss comparison CSVs, RMSE/MASE/MDA/NSE/KGE/skill curves, sensitivity heatmaps and summaries relative to the original composite, and **SHAP expected-gradient attributions for the composite-loss TCN and LSTM** (25 predictors, six feature groups, seven horizons, temporal-history heatmaps, and signed SHAP summary plots). Values express how much a feature history contributes to predicted ΔH **in feet**, relative to a training-period background. **SHAP is not a measurement of change in forecasting skill**; the loss sensitivity study still evaluates RMSE, MASE, MDA, NSE, KGE and persistence skill.

**Inference:** retained 100 MC-dropout passes for all eight experiment arms. For SHAP only, dropout is disabled, training windows supply the SHAP background, and sampled 2025 issue times are explained using gradients. The calculation does not alter fitted checkpoints. Rankings measure **model reliance**, not causal hydrologic influence. Correlated predictors can split importance; read the grouped rankings alongside individual rankings. The original composite is specified *before* examining 2025 as the source model for feature ranking. Do not use 2025 results for hyperparameter or model selection.

**Execution caveat:** Eight full models and 100 MC passes on each 2025 test set are substantial work even on a Colab GPU. Colab can disconnect. Results are written to Drive after each successful model; rerunning with the same settings verifies and reuses completed arms. If a run is interrupted before completion, rename its partial `TCN_fixed2025` or `LSTM_fixed2025` folder before retrying; the script never silently overwrites an incomplete run.
**Cache behavior:** Automatic download applies only to absent 2020–2025 source CSVs. Existing annual CSVs are reused unchanged; corrupted existing files are flagged for review rather than silently overwritten. A complete data preparation and leakage audit still precedes training. Download requires internet access. The `dataretrieval` dependency is installed by the notebook only when needed.

**Interpretation of SHAP:** Global rankings use the mean absolute value of a feature's **signed SHAP values summed across the 36 input timestamps**. Grouped SHAP is computed by summing signed feature contributions before taking absolute values. SHAP indicates model attribution rather than causal influence or test-skill deterioration. Correlated hydrologic predictors may divide attribution between features.


## 1. Mount Drive and configure the experiment

In Colab, select **Runtime → Change runtime type → T4 GPU** (or a faster GPU). Then set `USER_CACHE_DIR` to your Google Drive folder for the **18 annual raw USGS cache CSVs** named, e.g., `2020_02087275_00045_hw70_precip.csv`. With `DOWNLOAD_MISSING_CACHE=True`, the notebook **reuses any existing files and downloads only missing site/parameter/year CSVs** from USGS NWIS. If the folder is empty, it retrieves all 18 and saves them there; no changes to the fixed 2020–2025 experiment or 2025 test year. Set the flag to `False` to enforce strictly offline/cache-only processing. If USGS rejects a request, training stops instead of using a partial downloaded year.

`RUN_MODE = "prepare"` checks all files, splits, and scale without fitting models. `RUN_MODE = "all"` trains or reuses the selected losses and creates their outputs. `RUN_MODE = "report"` collects sensitivity results after all four objectives have completed without reloading the cache. For sessions with limited GPU time, set `RUN_LOSSES` to **one** objective per session, then process the remaining objectives in subsequent sessions; output hashes and settings are verified when existing arms are reused. Loss summary tables appear automatically once all four are complete.
**SHAP configuration:** `SHAP_EXPLAIN_ISSUES` sets the number of uniformly sampled 2025 windows; `SHAP_BACKGROUND_ISSUES` draws training windows as a baseline; `SHAP_GRADIENT_SAMPLES` controls the Monte Carlo effort per sample. SHAP runs after both composite-model checkpoints exist. Outputs are cached per architecture on Drive so an interrupted session can reuse valid attributions.


In [ ]:
from pathlib import Path
from google.colab import drive
import torch

drive.mount('/content/drive')
USER_CACHE_DIR = Path('/content/drive/MyDrive/usgs_cache')
USER_OUTPUT_ROOT = Path('/content/drive/MyDrive/crabtree_run_results_2026 Oct')
DOWNLOAD_MISSING_CACHE = True  # Automatically fetch only absent 2020–2025 USGS CSVs
RUN_MODE = 'all'  # 'prepare' | 'all' | 'report'
RUN_LOSSES = ('composite', 'RMSE', 'NSE', 'KGE')  # Or one loss per session, e.g. ('composite',)
RUN_FEATURE_IMPORTANCE = True
SHAP_EXPLAIN_ISSUES = 256            # Sampled 2025 issue times, not all test windows
SHAP_BACKGROUND_ISSUES = 64         # Training-period reference windows
SHAP_GRADIENT_SAMPLES = 64          # Monte Carlo draws for expected gradients
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('Raw cache:', USER_CACHE_DIR)
print('Outputs:', USER_OUTPUT_ROOT)
print('Auto-download missing cache:', DOWNLOAD_MISSING_CACHE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Torch: 2.11.0+cu130 | CUDA available: True
Raw cache: /content/drive/MyDrive/usgs_cache
Outputs: /content/drive/MyDrive/crabtree_run_results_2026 Oct
Auto-download missing cache: True


In [ ]:
# Install the USGS retrieval dependency only if observations are missing.
# Existing normalized caches remain untouched and require no network access.
import importlib.util
import subprocess
import sys
if RUN_MODE != 'report' and DOWNLOAD_MISSING_CACHE:
    need_download = bool([
        (year, site, cd, name)
        for year in range(2020, 2026)
        for site, cd, name in (
            ('02087275', '00045', 'hw70_precip'),
            ('02087275', '00065', 'hw70_stage'),
            ('02087324', '00065', 'us1_stage'),
        )
        if not (USER_CACHE_DIR / f'{year}_{site}_{cd}_{name}.csv').is_file()
    ])
    if need_download and importlib.util.find_spec('dataretrieval') is None:
        print('Installing USGS dataretrieval for missing cache files ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'dataretrieval'])
    else:
        print('USGS dependency download step not needed.')

# Install SHAP if this Colab runtime does not already have it.
# Analysis uses gradient-based SHAP, not permutation importance.
if RUN_MODE == 'all' and RUN_FEATURE_IMPORTANCE and importlib.util.find_spec('shap') is None:
    print('Installing SHAP for expected-gradient feature attribution ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'shap>=0.45,<0.52'])


USGS dependency download step not needed.


## 2. Model, loss functions, audit, comparison, and plot code

All implementation code is in this notebook. The fixed-2025 baseline design is retained, and the four loss objectives have independent model checkpoints and manifests. The code below defines the functions; it does not start fitting until section 3.

In [ ]:
"""Self-contained Colab implementation of the fixed-2025 TCN and LSTM comparison."""
from pathlib import Path
import argparse
import hashlib
import importlib.metadata
import json
import os
import platform
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

ROOT = Path.cwd()
LOCAL_TZ = "America/New_York"
HW70_SITE, US1_SITE = "02087275", "02087324"
PRECIP_CD, STAGE_CD = "00045", "00065"
START_YEAR, END_YEAR = 2020, 2025
DOWNLOAD_END_LOCAL = pd.Timestamp("2025-12-31", tz=LOCAL_TZ)
FORCE_DOWNLOAD = False
DOWNLOAD_RETRIES, RETRY_WAIT_SEC, REQUEST_PAUSE_SEC = 3, 3, 0.20
ALLOW_DOWNLOAD = False
CACHE_DIR = ROOT / "shared_usgs_cache"
TIMESTEP_MIN = 5
LOOKBACK_MIN = 180
LOOKBACK_STEPS = LOOKBACK_MIN // TIMESTEP_MIN
LAG_HYDRO_MIN, LAG_CREST_MIN = 30, 60
HORIZONS_MIN = [15, 30, 60, 90, 120, 150, 180]
HORIZON_STEPS = [h // TIMESTEP_MIN for h in HORIZONS_MIN]
TRAIN_YEARS, VAL_YEARS, TEST_YEARS = [2020, 2021, 2022, 2023], [2024], [2025]
TRAIN_SAMPLE_STRIDE = 1  # Authorized change: every valid 5-minute issue time.
MAX_EPOCHS, PATIENCE = 45, 8
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-5
DROPOUT = 0.15
HIGH_STAGE_FT = 4.0
RANDOM_SEED = 42
MC_DROPOUT_PASSES = 100
TARGET_TYPE = "delta_stage_ft"
DIRECTION_DEADBAND_FT = 0.05
DIRECTION_SOFT_TEMP_FT = 0.05
COMPOSITE_LOSS_MASE_WEIGHT = 0.50
COMPOSITE_LOSS_DIRECTION_WEIGHT = 0.50
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
if os.environ.get("HYDRO_REQUIRE_GPU") == "1" and DEVICE.type == "cpu":
    raise RuntimeError("GPU required but unavailable; refusing CPU training")
BATCH_SIZE = 512 if DEVICE.type in {"cuda", "mps"} else 128
ALIGNMENT_TOLERANCE = pd.Timedelta("2min30s")
FEATURE_NAMES = [
    "stage", "rain", "rain_available", "stage_lag30", "stage_lag60",
    "dStage_5", "dStage_15", "dStage_30", "stage_roll30_mean", "stage_roll60_max",
    "us1_stage", "us1_available", "us1_dStage_5", "us1_dStage_15", "us1_dStage_30",
    "rain_sum_15", "rain_cov_15", "rain_sum_30", "rain_cov_30",
    "rain_sum_60", "rain_cov_60", "rain_sum_120", "rain_cov_120",
    "rain_sum_180", "rain_cov_180",
]


def write_output_csv(df, name, index=False):
    path = OUTPUT_DIR / name
    df.to_csv(path, index=index)
    return path


# DATA LOADING — reuse normalized observation caches, never old interpolated data.
ARCHITECTURE_LABEL = "TCN"
OUTPUT_DIR = ROOT / "TCN_fixed2025"
TCN_CHANNELS = [32, 64, 64]
KERNEL_SIZE = 3
LSTM_HIDDEN_SIZE = 61
LSTM_NUM_LAYERS = 2

def as_utc_index(idx):
    """Return a timezone-aware UTC DatetimeIndex."""
    idx = pd.DatetimeIndex(pd.to_datetime(idx))
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")
    return idx

def local_year(index):
    """Calendar year in Raleigh local time, while keeping UTC internally."""
    idx = as_utc_index(index)
    return idx.tz_convert(LOCAL_TZ).year

def _pick_parameter_column(df: pd.DataFrame, parameter_cd: str):
    """
    Identify the numeric value column returned by dataretrieval.nwis.get_iv.

    Typical returned columns include the parameter code and a qualification
    column. We select the parameter-containing numeric column with the most
    usable observations, excluding qualification-code fields.
    """
    if df.empty:
        return None

    candidates = []
    for c in df.columns:
        s = str(c)
        sl = s.lower()

        if parameter_cd not in s:
            continue
        if sl.endswith("_cd") or "_cd_" in sl:
            continue
        if "qual" in sl:
            continue

        numeric_count = pd.to_numeric(df[c], errors="coerce").notna().sum()
        candidates.append((numeric_count, c))

    # Prefer exact parameter column if present.
    exact = [c for c in df.columns if str(c) == parameter_cd]
    if exact:
        return exact[0]

    if not candidates:
        # Last-resort scan of numeric columns.
        ignored = {"agency_cd", "site_no"}
        for c in df.columns:
            if str(c) in ignored:
                continue
            numeric_count = pd.to_numeric(df[c], errors="coerce").notna().sum()
            if numeric_count:
                candidates.append((numeric_count, c))

    if not candidates:
        return None

    candidates.sort(key=lambda z: z[0], reverse=True)
    return candidates[0][1]

def _normalize_nwis_frame(df, site, parameter_cd, output_name):
    """
    Normalize dataretrieval NWIS output to:
        UTC datetime index + one numeric value column.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=[output_name],
                            index=pd.DatetimeIndex([], tz="UTC"))

    df = df.copy()

    # dataretrieval normally supplies a datetime index.
    if isinstance(df.index, pd.MultiIndex):
        # If a multi-index unexpectedly appears, choose the datetime level.
        dt_level = None
        for lev in range(df.index.nlevels):
            vals = df.index.get_level_values(lev)
            if pd.api.types.is_datetime64_any_dtype(pd.Series(vals).dtype):
                dt_level = lev
                break
        if dt_level is None:
            # Try parsing each level.
            for lev in range(df.index.nlevels):
                parsed = pd.to_datetime(df.index.get_level_values(lev), errors="coerce")
                if pd.Series(parsed).notna().mean() > 0.9:
                    dt_level = lev
                    break
        if dt_level is None:
            raise ValueError("Could not identify datetime level in NWIS MultiIndex.")
        idx = pd.DatetimeIndex(df.index.get_level_values(dt_level))
    elif isinstance(df.index, pd.DatetimeIndex):
        idx = df.index
    elif "datetime" in df.columns:
        idx = pd.DatetimeIndex(pd.to_datetime(df["datetime"], errors="coerce"))
    else:
        parsed = pd.to_datetime(df.index, errors="coerce")
        if pd.Series(parsed).notna().mean() < 0.9:
            raise ValueError("Could not identify a datetime index in NWIS response.")
        idx = pd.DatetimeIndex(parsed)

    idx = as_utc_index(idx)

    value_col = _pick_parameter_column(df, parameter_cd)
    if value_col is None:
        print(f"  [WARN] No value column found for site={site}, parameter={parameter_cd}")
        return pd.DataFrame(columns=[output_name],
                            index=pd.DatetimeIndex([], tz="UTC"))

    values = pd.to_numeric(df[value_col], errors="coerce").to_numpy()

    out = pd.DataFrame({output_name: values}, index=idx)
    out.index.name = "datetime"
    out = out[~out.index.duplicated(keep="last")].sort_index()

    return out

def _query_nwis_once(site, parameter_cd, output_name, start_date, end_date):
    """One instantaneous-value NWIS request."""
    from dataretrieval import nwis  # Optional: needed only for an explicit download.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        df, _metadata = nwis.get_iv(
            sites=site,
            start=str(start_date),
            end=str(end_date),
            parameterCd=parameter_cd,
            multi_index=False,
        )
    return _normalize_nwis_frame(
        df=df,
        site=site,
        parameter_cd=parameter_cd,
        output_name=output_name,
    )

def _query_with_retries(site, parameter_cd, output_name, start_date, end_date):
    last_exc = None

    for attempt in range(1, DOWNLOAD_RETRIES + 1):
        try:
            return _query_nwis_once(
                site, parameter_cd, output_name, start_date, end_date
            )
        except Exception as exc:
            last_exc = exc
            print(
                f"    request attempt {attempt}/{DOWNLOAD_RETRIES} failed: {exc}"
            )
            if attempt < DOWNLOAD_RETRIES:
                time.sleep(RETRY_WAIT_SEC * attempt)

    raise RuntimeError(
        f"NWIS request failed after {DOWNLOAD_RETRIES} attempts: "
        f"site={site}, parameter={parameter_cd}, "
        f"{start_date} to {end_date}"
    ) from last_exc

def _monthly_fallback(site, parameter_cd, output_name, year, final_end):
    """
    If a full-year IV request fails, retry in calendar-month chunks.
    Any failed monthly request invalidates the year rather than silently omitting it.
    """
    pieces = []
    failed_months = []

    y_start = pd.Timestamp(f"{year}-01-01")
    y_end = pd.Timestamp(final_end)

    current = y_start

    while current <= y_end:
        month_end = (current + pd.offsets.MonthEnd(0)).normalize()
        chunk_end = min(month_end, y_end)

        print(
            f"      monthly fallback: {current.date()} -> {chunk_end.date()}"
        )

        try:
            part = _query_with_retries(
                site, parameter_cd, output_name,
                current.date(), chunk_end.date()
            )
            if not part.empty:
                pieces.append(part)
        except Exception as exc:
            failed_months.append(f"{current.date()} to {chunk_end.date()}: {exc}")
            print(f"      [ERROR] monthly chunk failed: {exc}")

        current = (chunk_end + pd.Timedelta(days=1)).normalize()
        time.sleep(REQUEST_PAUSE_SEC)

    if failed_months:
        raise RuntimeError(
            f"USGS monthly fallback incomplete for {site}/{parameter_cd}/{year}; "
            "refusing to cache a partial year. Failed chunks:\n" +
            "\n".join(failed_months)
        )
    if not pieces:
        return pd.DataFrame(columns=[output_name],
                            index=pd.DatetimeIndex([], tz="UTC"))

    out = pd.concat(pieces)
    out = out[~out.index.duplicated(keep="last")].sort_index()
    return out

def download_year_series(site, parameter_cd, output_name, year):
    """
    Download one site/parameter/year and cache it as CSV.

    First tries one annual request. If that fails, it automatically switches
    to monthly chunks. Cached files are reused on later runs.
    """
    cache_file = CACHE_DIR / f"{year}_{site}_{parameter_cd}_{output_name}.csv"

    if cache_file.exists() and not FORCE_DOWNLOAD:
        cached = pd.read_csv(cache_file)
        if "datetime" not in cached.columns and "index" in cached.columns:
            cached = cached.rename(columns={"index": "datetime"})
        if "datetime" not in cached.columns:
            raise ValueError(
                f"Cache {cache_file.name} missing datetime column; "
                "delete it to force re-download."
            )
        cached["datetime"] = pd.to_datetime(cached["datetime"], utc=True)
        idx = as_utc_index(cached["datetime"])
        out = pd.DataFrame(
            {output_name: pd.to_numeric(cached[output_name], errors="coerce").values},
            index=idx
        )
        out.index.name = "datetime"
        print(f"  [CACHE] {cache_file.name}: {len(out):,} rows")
        return out

    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError(f"Missing raw cache: {cache_file}. Supply --cache-dir or enable download_missing=True.")

    start_local = pd.Timestamp(f"{year}-01-01", tz=LOCAL_TZ)

    if year == END_YEAR:
        end_local = DOWNLOAD_END_LOCAL
    else:
        end_local = pd.Timestamp(f"{year}-12-31", tz=LOCAL_TZ)

    if start_local > DOWNLOAD_END_LOCAL:
        return pd.DataFrame(columns=[output_name],
                            index=pd.DatetimeIndex([], tz="UTC"))

    print(
        f"  Downloading {output_name:<12s} | site {site} | "
        f"{start_local.date()} -> {end_local.date()}"
    )

    try:
        out = _query_with_retries(
            site, parameter_cd, output_name,
            start_local.date(), end_local.date(),
        )
        if out.empty or not np.isfinite(out[output_name].to_numpy(dtype=float)).any():
            raise ValueError("Annual response contains no usable observations")
    except Exception as exc:
        print(f"    [WARN] annual request failed or returned no usable data: {exc}")
        print("    Switching to monthly fallback ...")
        out = _monthly_fallback(site, parameter_cd, output_name, year, end_local.date())

    if out.empty or not np.isfinite(out[output_name].to_numpy(dtype=float)).any():
        raise RuntimeError(
            f"USGS returned no usable {output_name} observations for site {site} in {year}. "
            "Nothing was cached; training will not proceed."
        )

    out = out.copy()
    out.index.name = "datetime"
    # Write a temporary file and rename only after a complete successful download.
    # A failed/disconnected Colab run must not leave an apparently finished CSV.
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    temporary = cache_file.with_name(cache_file.name + ".partial")
    try:
        out.reset_index().to_csv(temporary, index=False)
        os.replace(temporary, cache_file)
    finally:
        temporary.unlink(missing_ok=True)

    print(f"    saved {len(out):,} observations -> {cache_file.name}")

    time.sleep(REQUEST_PAUSE_SEC)
    return out

# CAUSAL PREPROCESSING

def align_past_only(frame, index, name):
    """Latest source timestamp <= grid time, with at most 2.5 minutes of age.

    No interpolation, backward fill, centered resampling, or future match.
    An explicitly missing source observation remains missing.
    """
    frame = frame.copy()
    frame.index = as_utc_index(frame.index)
    frame = frame[~frame.index.duplicated(keep="last")].sort_index()
    if frame.empty:
        return pd.Series(np.nan, index=index, name=name)
    source = frame[[name]].copy()
    source["source_time"] = source.index
    aligned = source.reindex(index, method="ffill", tolerance=ALIGNMENT_TOLERANCE)
    matched = aligned["source_time"].notna()
    grid_times = pd.Series(index, index=index)
    ages = grid_times[matched] - aligned.loc[matched, "source_time"]
    if (ages < pd.Timedelta(0)).any() or (ages > ALIGNMENT_TOLERANCE).any():
        raise ValueError(f"Noncausal/out-of-tolerance source timestamp for {name}")
    return pd.to_numeric(aligned[name], errors="raise").rename(name)


def load_causal_data():
    specs = [(HW70_SITE, PRECIP_CD, "hw70_precip"),
             (HW70_SITE, STAGE_CD, "hw70_stage"),
             (US1_SITE, STAGE_CD, "us1_stage")]
    index = pd.date_range(
        pd.Timestamp("2020-01-01", tz=LOCAL_TZ).tz_convert("UTC"),
        pd.Timestamp("2026-01-01", tz=LOCAL_TZ).tz_convert("UTC"),
        freq="5min", inclusive="left")
    raw = pd.DataFrame(index=index)
    raw.index.name = "datetime_utc"
    cache_hashes = {}
    for site, parameter, name in specs:
        parts = []
        for year in range(START_YEAR, END_YEAR + 1):
            part = download_year_series(site, parameter, name, year)
            parts.append(part)
            path = CACHE_DIR / f"{year}_{site}_{parameter}_{name}.csv"
            if not path.exists():
                raise ValueError(f"Missing raw cache after loading: {path}")
            cache_hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
        source = pd.concat(parts)
        raw[name] = align_past_only(source, index, name)
    raw["rain_available"] = raw["hw70_precip"].notna().astype(float)
    if np.isinf(raw.to_numpy(dtype=float)).any():
        raise ValueError("Infinite raw observations; inspect source data before training")
    # Keep targets un-interpolated too: missing issue/future labels invalidate samples.
    return raw, cache_hashes


# FEATURE ENGINEERING — preserved ordered 25 predictors.
def build_causal_features(df: pd.DataFrame) -> pd.DataFrame:
    """HW70 + lag + US-1 features; input must already be causally aligned."""
    x = pd.DataFrame(index=df.index)

    # Current upstream state
    x["stage"] = df["hw70_stage"]
    x["rain"] = df["hw70_precip"]
    x["rain_available"] = df["rain_available"]

    # Explicit routing lags (past only: shift looks backward).
    x["stage_lag30"] = df["hw70_stage"].shift(LAG_HYDRO_MIN // TIMESTEP_MIN)
    x["stage_lag60"] = df["hw70_stage"].shift(LAG_CREST_MIN // TIMESTEP_MIN)

    # Causal stage derivatives (how fast HW70 is rising now)
    x["dStage_5"] = df["hw70_stage"].diff(1)
    x["dStage_15"] = df["hw70_stage"].diff(3)
    x["dStage_30"] = df["hw70_stage"].diff(6)

    # Stage context
    x["stage_roll30_mean"] = df["hw70_stage"].rolling(6, min_periods=1).mean()
    x["stage_roll60_max"] = df["hw70_stage"].rolling(12, min_periods=1).max()

    # Current US-1 state at issue time (short-lead skill).
    x["us1_stage"] = df["us1_stage"]
    x["us1_available"] = df["us1_stage"].notna().astype(np.float32)
    x["us1_dStage_5"] = df["us1_stage"].diff(1)
    x["us1_dStage_15"] = df["us1_stage"].diff(3)
    x["us1_dStage_30"] = df["us1_stage"].diff(6)

    # Rainfall totals with explicit completeness.
    # If a rain window is incomplete, total is NaN and coverage is supplied.
    for win_min in [15, 30, 60, 120, 180]:
        k = win_min // TIMESTEP_MIN
        cnt = df["hw70_precip"].rolling(k, min_periods=1).count()
        sm = df["hw70_precip"].rolling(k, min_periods=1).sum()

        x[f"rain_sum_{win_min}"] = sm.where(cnt == k)
        x[f"rain_cov_{win_min}"] = cnt / float(k)

    return x

# FULL-DATA SEQUENCE CONSTRUCTION
class SequenceDataset(Dataset):
    def __init__(
        self,
        feature_df,
        target_series,
        scaler,
        lookback_steps,
        horizon_steps,
        fit_scaler=False,
        sample_stride=1,
    ):
        if list(feature_df.columns) != FEATURE_NAMES:
            raise ValueError("Ordered feature inventory mismatch")
        if not feature_df.index.equals(target_series.index):
            raise ValueError("Predictor/target index mismatch")
        if not feature_df.index.is_unique or not feature_df.index.is_monotonic_increasing:
            raise ValueError("Sequence timeline must be unique and increasing")
        if len(feature_df) > 1 and not np.all(np.diff(feature_df.index.asi8) == pd.Timedelta(minutes=5).value):
            raise ValueError("Sequences require an uninterrupted 5-minute grid")
        self.features = feature_df.copy()
        self.target = target_series.copy()
        self.lookback = lookback_steps
        self.horizon_steps = horizon_steps

        # Missing-input handling:
        # - NaNs are replaced AFTER adding availability/coverage channels.
        # - no stage interpolation is performed; original validity rules apply.
        # - zero AFTER standardization is the training feature mean, not zero rain.
        #   Existing rain coverage and US-1 availability flags are preserved.
        X = self.features.values.astype(np.float32)

        if fit_scaler:
            scaler.fit(X)

        Xs = scaler.transform(X).astype(np.float32)
        Xs = np.nan_to_num(Xs, nan=0.0, posinf=0.0, neginf=0.0)

        self.X = Xs
        self.y = self.target.values.astype(np.float32)
        self.index = self.features.index

        max_h = max(horizon_steps)

        valid = []
        for i in range(
            self.lookback - 1,
            len(self.features) - max_h,
            max(1, int(sample_stride))
        ):
            # Current upstream stage must exist.
            if not np.isfinite(self.features["stage"].iloc[i]):
                continue

            # Persistence benchmark needs the current US-1 stage at issue time.
            if not np.isfinite(self.y[i]):
                continue

            ys = [self.y[i + h] for h in horizon_steps]
            if not np.all(np.isfinite(ys)):
                continue

            # Need enough finite upstream stage values in the lookback.
            stage_window = self.features["stage"].iloc[
                i - self.lookback + 1:i + 1
            ].values
            if np.isfinite(stage_window).mean() < 0.95:
                continue

            valid.append(i)

        self.valid = np.asarray(valid, dtype=int)

    def __len__(self):
        return len(self.valid)

    def __getitem__(self, j):
        i = self.valid[j]
        start = i - self.lookback + 1
        x = self.X[start:i + 1]  # [T, F]
        issue_stage = self.y[i]

        # ΔH TARGET: future US-1 minus issue-time US-1 (not absolute stage).
        y_delta = np.asarray(
            [self.y[i + h] - issue_stage for h in self.horizon_steps],
            dtype=np.float32
        )

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y_delta, dtype=torch.float32),
            torch.tensor(issue_stage, dtype=torch.float32),
            i,
        )

    def issue_times(self):
        return self.index[self.valid]

def make_sequence_dataset(feature_df, target_series, scaler, sample_stride=1):
    """SequenceDataset with publication lookback/horizons; scaler already fit."""
    return SequenceDataset(
        feature_df,
        target_series,
        scaler,
        LOOKBACK_STEPS,
        HORIZON_STEPS,
        fit_scaler=False,
        sample_stride=sample_stride,
    )

# TRAIN/VALIDATION/TEST SPLIT — local calendar years, UTC timestamps.
def split_by_calendar_year(raw):
    yrs = local_year(raw.index)

    train = raw[np.isin(yrs, TRAIN_YEARS)].copy()
    val = raw[np.isin(yrs, VAL_YEARS)].copy()
    test = raw[np.isin(yrs, TEST_YEARS)].copy()

    if train.empty or val.empty or test.empty:
        raise RuntimeError(
            "A publication split is empty. Check downloaded coverage for "
            "2020-2025."
        )

    return train, val, test

# MODEL — unchanged controlled architecture.
class Chomp1d(nn.Module):
    """
    Remove right padding to make Conv1d strictly causal.
    """
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size]

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.net = nn.Sequential(
            nn.Conv1d(
                in_ch, out_ch, kernel_size,
                padding=padding,
                dilation=dilation
            ),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(
                out_ch, out_ch, kernel_size,
                padding=padding,
                dilation=dilation
            ),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.downsample = (
            nn.Conv1d(in_ch, out_ch, 1)
            if in_ch != out_ch else None
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class HydroTCN(nn.Module):
    def __init__(self, n_features, channels, kernel_size, dropout, n_outputs):
        super().__init__()

        blocks = []
        in_ch = n_features

        for level, out_ch in enumerate(channels):
            dilation = 2 ** level
            blocks.append(
                TemporalBlock(
                    in_ch,
                    out_ch,
                    kernel_size,
                    dilation,
                    dropout,
                )
            )
            in_ch = out_ch

        self.tcn = nn.Sequential(*blocks)

        self.head = nn.Sequential(
            nn.Linear(channels[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_outputs),
        )

    def forward(self, x):
        # x: [B, T, F] -> Conv1d expects [B, F, T]
        x = x.transpose(1, 2)
        z = self.tcn(x)

        # Last causal representation at issue time.
        z_last = z[:, :, -1]
        return self.head(z_last)

class HydroLSTM(nn.Module):
    """
    Parameter-matched recurrent comparator for the HydroTCN.

    Fair-comparison rules:
      - identical [B, T, F] causal input tensor
      - identical 180-min lookback and seven forecast horizons
      - identical multi-horizon ΔH target and prediction head width
      - identical loss, sampler, optimizer, split, scaling, early stopping,
        persistence benchmark, and MC-pass count
      - only the temporal encoder is changed from dilated causal convolutions
        to a stacked LSTM
    """
    def __init__(
        self,
        n_features,
        hidden_size,
        num_layers,
        dropout,
        n_outputs,
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # Keep the same 64-unit prediction head used by the TCN.
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_outputs),
        )

    def forward(self, x):
        # x: [B, T, F]. LSTM consumes the sequence in chronological order.
        z, _ = self.lstm(x)

        # Representation available at the issue time only (last causal step).
        z_last = z[:, -1, :]
        return self.head(z_last)


# TRAINING — original loss, weighted sampler, optimizer and stopping rule.
def timing_mase_direction_loss(
    pred_delta,
    target_delta,
    mase_scale,
    deadband_ft=DIRECTION_DEADBAND_FT,
    direction_temp_ft=DIRECTION_SOFT_TEMP_FT,
):
    """
    Composite differentiable objective aligned with the study question.

    Component A — MASE-aligned magnitude term:
      raw_mase = mean(|predicted ΔH - observed ΔH|) / training-only naive MAE.
      Because raw MASE can be numerically much larger than a 0..1 direction term,
      use the monotone bounded transform raw_mase/(1+raw_mase) inside the loss.
      Exact, untransformed MASE is still reported during evaluation.

    Component B — differentiable MDA-style direction term:
      raw MDA uses sign / class decisions and is not differentiable, so it cannot
      be back-propagated directly. We use a smooth wrong-direction probability.
      For observed rises/falls outside the deadband it rewards the correct sign;
      for stable targets it rewards predictions that stay close to zero.

    Both architectures use this exact same objective. The loss is intentionally
    balanced 50/50 after putting both components on approximately 0..1 scales.
    """
    scale = float(mase_scale)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError(f"mase_scale must be finite and >0, got {mase_scale}")

    eps = 1e-6
    err = pred_delta - target_delta

    # Exact MASE numerator divided by TRAINING-only one-step naive MAE.
    raw_mase = torch.mean(torch.abs(err)) / max(scale, eps)
    bounded_mase = raw_mase / (1.0 + raw_mase)

    # Smooth surrogate for direction accuracy / MDA.
    deadband = max(float(deadband_ft), eps)
    temp = max(float(direction_temp_ft), eps)
    target_abs = torch.abs(target_delta)
    meaningful = target_abs > deadband
    target_sign = torch.sign(target_delta)

    # Rise/fall: 0 = confidently correct sign, 1 = confidently wrong sign.
    directional_wrong_prob = torch.sigmoid(
        -(target_sign * pred_delta) / temp
    )

    # Stable target: 0 at ΔH_pred=0 and smoothly approaches 1 as |prediction| grows.
    stable_wrong_prob = 1.0 - torch.exp(
        -torch.square(pred_delta / deadband)
    )

    soft_mda_error = torch.where(
        meaningful, directional_wrong_prob, stable_wrong_prob
    ).mean()

    total = (
        COMPOSITE_LOSS_MASE_WEIGHT * bounded_mase
        + COMPOSITE_LOSS_DIRECTION_WEIGHT * soft_mda_error
    )
    return total, {
        "raw_mase": raw_mase.detach(),
        "bounded_mase": bounded_mase.detach(),
        "soft_mda_error": soft_mda_error.detach(),
    }

LOSS_NAMES = ("composite", "RMSE", "NSE", "KGE")
TRAINING_LOSS_DESCRIPTIONS = {
    "composite": "0.5*bounded_training_MASE + 0.5*soft_MDA_error",
    "RMSE": "sqrt(mean_batch,horizon((predicted_delta-observed_delta)^2)+eps)",
    "NSE": "mean_horizon(mean_batch(error^2)/training_stage_variance_horizon)",
    "KGE": "mean_horizon(1-KGE_2009(batch reconstructed absolute stage))",
}


def training_reference_from_dataset(ds):
    """Observed train-only stage variance for each horizon; excludes val and test."""
    valid = ds.valid
    if len(valid) < 2:
        raise ValueError("Insufficient training sequences for NSE reference")
    obs = np.stack([ds.y[valid + h] for h in HORIZON_STEPS], axis=1).astype(float)
    if not np.isfinite(obs).all():
        raise ValueError("Missing training stage values for NSE denominator")
    variances = np.var(obs, axis=0)
    if np.any(variances <= 1e-8):
        raise ValueError("Near-constant training stage; NSE objective is ill-defined")
    return {"stage_variance_by_horizon_ft2": variances.tolist(),
            "train_stage_std_ft": float(np.sqrt(np.mean(variances))),
            "n_training_sequences": int(len(valid))}


def select_training_loss(pred_delta, target_delta, issue_stage, mase_scale, name, reference):
    """Compute loss on shared ΔH forecasts; KGE uses reconstructed stage.

    NSE training uses a *fixed training-observation* denominator to avoid
    random-batch zero-variance explosions. Reported test NSE is conventional,
    computed relative to each test horizon's observed-stage mean.
    KGE is the 2009 r/alpha/beta formulation and is computed per horizon.
    """
    composite, comp = timing_mase_direction_loss(pred_delta, target_delta, mase_scale)
    if name == "composite":
        if not torch.isfinite(composite):
            raise FloatingPointError("Non-finite original composite loss")
        return composite, comp
    if name == "RMSE":
        loss = torch.sqrt(torch.mean((pred_delta - target_delta).square()) + 1e-10)
    elif name == "NSE":
        variance = torch.as_tensor(reference["stage_variance_by_horizon_ft2"],
                                   device=pred_delta.device, dtype=pred_delta.dtype)
        loss = ((pred_delta - target_delta).square().mean(dim=0) / variance).mean()
    elif name == "KGE":
        if pred_delta.shape[0] < 4:
            # Very last tiny batch has undefined correlations. Use a stable
            # training-reference-scaled squared error for just that tail.
            variance = torch.as_tensor(reference["stage_variance_by_horizon_ft2"],
                                      device=pred_delta.device, dtype=pred_delta.dtype)
            loss = ((pred_delta-target_delta).square().mean(dim=0)/variance).mean()
            return loss, comp
        observed = issue_stage[:, None] + target_delta
        forecast = issue_stage[:, None] + pred_delta
        om, pm = observed.mean(dim=0), forecast.mean(dim=0)
        oc, pc = observed - om, forecast - pm
        # Scale-aware regularization avoids undefined correlation in flat batches.
        eps_std = max(float(reference["train_stage_std_ft"]) * 1e-3, 1e-5)
        so = torch.sqrt(oc.square().mean(dim=0) + eps_std**2)
        sp = torch.sqrt(pc.square().mean(dim=0) + eps_std**2)
        r = (oc * pc).mean(dim=0) / (so * sp)
        alpha = sp / so
        beta = pm / torch.where(om.abs() < 1e-4,
                                 torch.where(om >= 0, torch.full_like(om, 1e-4),
                                             torch.full_like(om, -1e-4)), om)
        loss = torch.sqrt((r - 1).square() + (alpha - 1).square() +
                          (beta - 1).square() + 1e-10).mean()
    else:
        raise ValueError(f"Unknown loss {name}")
    if not torch.isfinite(loss):
        raise FloatingPointError(f"Non-finite {name} loss; inspect target variability")
    return loss, comp


def storm_sample_weights(ds):
    """
    Train more often on rising / high-water issue times so long-lead
    peaks are actually seen.
    """
    w = np.ones(len(ds), dtype=np.float64)
    d30 = ds.features["dStage_30"].values
    for j, i in enumerate(ds.valid):
        y0 = ds.y[i]
        rise = d30[i] if i < len(d30) else 0.0
        if np.isfinite(y0) and y0 >= HIGH_STAGE_FT:
            w[j] = 8.0
        elif np.isfinite(rise) and rise > 0.15:
            w[j] = 4.0
    return w

def set_global_seed(seed):
    """Reset NumPy/PyTorch RNGs before each independent training run."""
    seed = int(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        if hasattr(torch, "mps") and hasattr(torch.mps, "manual_seed"):
            torch.mps.manual_seed(seed)

def train_model(
    train_ds,
    val_ds,
    n_features,
    mase_scale,
    seed=RANDOM_SEED,
    loss_name="composite",
    train_reference=None,
    max_epochs=None,
    patience=None,
    save_artifacts=True,
    artifact_prefix="",
    verbose=True,
):
    """Train one TCN/LSTM run with the selected objective.

    Fixed study defaults retain 45 epochs maximum and patience of 8.
    """
    set_global_seed(seed)
    max_epochs = MAX_EPOCHS if max_epochs is None else int(max_epochs)
    patience = PATIENCE if patience is None else int(patience)

    if loss_name not in LOSS_NAMES:
        raise ValueError(f"Unsupported training loss {loss_name!r}")
    if loss_name in {"NSE", "KGE"} and train_reference is None:
        raise ValueError(f"{loss_name} training requires training-data reference statistics")
    train_weights = storm_sample_weights(train_ds)
    gen = torch.Generator()
    gen.manual_seed(int(seed))
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(train_weights, dtype=torch.double),
        num_samples=len(train_ds),
        replacement=True,
        generator=gen,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )

    if ARCHITECTURE_LABEL == "TCN":
        model = HydroTCN(
            n_features=n_features, channels=TCN_CHANNELS,
            kernel_size=KERNEL_SIZE, dropout=DROPOUT,
            n_outputs=len(HORIZONS_MIN),
        ).to(DEVICE)
    elif ARCHITECTURE_LABEL == "LSTM":
        model = HydroLSTM(
            n_features=n_features, hidden_size=LSTM_HIDDEN_SIZE,
            num_layers=LSTM_NUM_LAYERS, dropout=DROPOUT,
            n_outputs=len(HORIZONS_MIN),
        ).to(DEVICE)
    else:
        raise ValueError(f"Unknown architecture: {ARCHITECTURE_LABEL}")

    if verbose:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"{ARCHITECTURE_LABEL} trainable parameters: {n_params:,}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_val = np.inf
    patience_count = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses, train_mase, train_dir = [], [], []
        for xb, yb, sb, _ in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            sb = sb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss, comp = select_training_loss(
                pred, yb, sb, mase_scale, loss_name, train_reference
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())
            train_mase.append(float(comp["raw_mase"].cpu()))
            train_dir.append(float(comp["soft_mda_error"].cpu()))

        model.eval()
        val_losses, val_mase, val_dir = [], [], []
        with torch.no_grad():
            for xb, yb, sb, _ in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                sb = sb.to(DEVICE)
                pred = model(xb)
                loss, comp = select_training_loss(
                    pred, yb, sb, mase_scale, loss_name, train_reference
                )
                val_losses.append(loss.item())
                val_mase.append(float(comp["raw_mase"].cpu()))
                val_dir.append(float(comp["soft_mda_error"].cpu()))

        tr_loss = float(np.mean(train_losses))
        va_loss = float(np.mean(val_losses))
        history.append({
            "epoch": epoch,
            "train_loss": tr_loss,
            "val_loss": va_loss,
            "train_raw_mase": float(np.mean(train_mase)),
            "val_raw_mase": float(np.mean(val_mase)),
            "train_soft_mda_error": float(np.mean(train_dir)),
            "val_soft_mda_error": float(np.mean(val_dir)),
            "seed": int(seed),
        })

        if verbose:
            print(
                f"Seed {seed} | Epoch {epoch:03d} | "
                f"train={tr_loss:.5f} | val={va_loss:.5f} | "
                f"val_MASE={np.mean(val_mase):.3f} | "
                f"val_softMDAerr={np.mean(val_dir):.3f}"
            )

        if va_loss < best_val - 1e-5:
            best_val = va_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)
    if save_artifacts:
        hist_name = f"{artifact_prefix}training_history.csv"
        write_output_csv(history_df, hist_name)
        torch.save(
            {
                "model_state": model.state_dict(),
                "n_features": n_features,
                "feature_names": list(train_ds.features.columns),
                "horizons_min": HORIZONS_MIN,
                "lookback_min": LOOKBACK_MIN,
                "target_type": TARGET_TYPE,
        "training_loss": TRAINING_LOSS_DESCRIPTIONS[loss_name],
        "training_loss_note": (
            "Composite uses bounded MASE and a differentiable directional surrogate. "
            "RMSE uses delta error, NSE uses train-only stage variance weights, "
            "and KGE uses regularized batch KGE2009 on absolute stage. "
            "Every test metric uses 2025 observations; exact raw MDA is not a trainable loss."
        ),
                "mc_dropout_passes": MC_DROPOUT_PASSES,
                "architecture": ARCHITECTURE_LABEL,
                "seed": int(seed),
                "loss_name": loss_name,
                "timing_first_redesign": loss_name == "composite",
                "training_objective": TRAINING_LOSS_DESCRIPTIONS[loss_name],
                "mase_scale_training_ft": float(mase_scale),
                "direction_deadband_ft": float(DIRECTION_DEADBAND_FT),
                "direction_soft_temperature_ft": float(DIRECTION_SOFT_TEMP_FT),
            },
            OUTPUT_DIR / f"{artifact_prefix}hydro_{ARCHITECTURE_LABEL.lower()}_delta_mc.pt",
        )

    return model, history_df

# PREDICTION — retain original 100-pass MC mean for the main fixed test.
def enable_mc_dropout_only(model):
    """
    MC-DROPOUT INFERENCE: keep BatchNorm/Conv in eval mode but turn Dropout on.
    Only nn.Dropout modules are stochastic; this approximates epistemic uncertainty.
    """
    model.eval()
    for module in model.modules():
        if isinstance(module, nn.Dropout) or (ARCHITECTURE_LABEL == "LSTM" and isinstance(module, nn.LSTM)):
            module.train()

def reconstruct_absolute_stage(issue_stage, delta_arr):
    """ABSOLUTE RECONSTRUCTION: H_hat(t+h) = H(t) + ΔH_hat."""
    issue = np.asarray(issue_stage, dtype=np.float64)
    delta = np.asarray(delta_arr, dtype=np.float64)
    if delta.ndim == 1:
        return issue + delta
    return issue[:, None] + delta

def predict_dataset_mc(model, ds, n_passes=MC_DROPOUT_PASSES):
    """
    Batch inference with:
      - one deterministic dropout-off ΔH pass (diagnostics)
      - n_passes stochastic dropout-on ΔH passes -> MC mean / std / percentiles
    Absolute stage stats are reconstructed by adding issue-time US-1 stage.
    """
    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )

    det_delta = []
    mc_mean_delta = []
    mc_std_delta = []
    mc_p025_delta = []
    mc_p50_delta = []
    mc_p975_delta = []
    obs_delta = []
    issue_stage = []
    indices = []

    model.eval()

    with torch.no_grad():
        for xb, yb, sb, ib in loader:
            xb = xb.to(DEVICE)
            sb_np = sb.numpy()

            # Deterministic (dropout off)
            det = model(xb).cpu().numpy()
            det_delta.append(det)

            # MC-DROPOUT: enable Dropout only, repeat forward passes
            enable_mc_dropout_only(model)
            passes = []
            for _ in range(n_passes):
                passes.append(model(xb).cpu().numpy())
            model.eval()

            stack = np.stack(passes, axis=0)  # [passes, batch, horizons]
            mc_mean_delta.append(stack.mean(axis=0))
            mc_std_delta.append(stack.std(axis=0, ddof=0))
            mc_p025_delta.append(np.percentile(stack, 2.5, axis=0))
            mc_p50_delta.append(np.percentile(stack, 50.0, axis=0))
            mc_p975_delta.append(np.percentile(stack, 97.5, axis=0))

            obs_delta.append(yb.numpy())
            issue_stage.append(sb_np)
            indices.append(ib.numpy())

    det_delta = np.concatenate(det_delta, axis=0)
    mc_mean_delta = np.concatenate(mc_mean_delta, axis=0)
    mc_std_delta = np.concatenate(mc_std_delta, axis=0)
    mc_p025_delta = np.concatenate(mc_p025_delta, axis=0)
    mc_p50_delta = np.concatenate(mc_p50_delta, axis=0)
    mc_p975_delta = np.concatenate(mc_p975_delta, axis=0)
    obs_delta = np.concatenate(obs_delta, axis=0)
    issue_stage = np.concatenate(issue_stage, axis=0)
    indices = np.concatenate(indices, axis=0)
    times = ds.index[indices]

    return {
        "times": times,
        "obs_delta": obs_delta,
        "issue_stage": issue_stage,
        "det_delta": det_delta,
        "mc_mean_delta": mc_mean_delta,
        "mc_std_delta": mc_std_delta,
        "mc_p025_delta": mc_p025_delta,
        "mc_p50_delta": mc_p50_delta,
        "mc_p975_delta": mc_p975_delta,
    }

# MASE — retained training-only one-step (5-minute) scale.
def mase_scale_from_training(target_series):
    """Standard non-seasonal MASE scale from TRAINING data only.

    This is mean absolute one-step naive error in the training series. It is
    deliberately separate from horizon-specific persistence skill.
    """
    y = np.asarray(target_series, dtype=float)
    if len(y) < 2:
        return np.nan
    a, b = y[:-1], y[1:]
    good = np.isfinite(a) & np.isfinite(b)
    if not good.any():
        return np.nan
    scale = float(np.mean(np.abs(b[good] - a[good])))
    return scale if scale > 0 else np.nan

# MDA — raw sign agreement; exact zero is a separate sign, no deadband.

def primary_metrics(predictions, scale):
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("MASE scale must be finite and positive")
    rows = []
    for horizon, group in predictions.groupby("horizon_min", sort=True):
        observed = group["observed_delta_h"].to_numpy(float)
        predicted = group["predicted_delta_h"].to_numpy(float)
        if not np.isfinite(observed).all() or not np.isfinite(predicted).all():
            raise ValueError("Nonfinite prediction/target: do not silently drop samples")
        rows.append({"horizon_min": int(horizon),
                     "MASE": float(np.mean(np.abs(predicted-observed)) / scale),
                     "MDA": float(np.mean(np.sign(predicted) == np.sign(observed)))})
    result = pd.DataFrame(rows)
    if result.empty or result["horizon_min"].tolist() != HORIZONS_MIN:
        raise ValueError("Missing forecast horizon")
    return result


# OUTPUT

def prediction_table(mc_out):
    """All seven predictions from each common valid issue time; stage in feet."""
    frames = []
    issue = np.asarray(mc_out["issue_stage"], dtype=np.float64)
    for j, horizon in enumerate(HORIZONS_MIN):
        obs = np.asarray(mc_out["obs_delta"][:, j], dtype=np.float64)
        pred = np.asarray(mc_out["mc_mean_delta"][:, j], dtype=np.float64)
        frames.append(pd.DataFrame({
            "issue_time": mc_out["times"],
            "valid_time": mc_out["times"] + pd.Timedelta(minutes=horizon),
            "horizon_min": horizon,
            "observed_stage": issue + obs,
            "observed_delta_h": obs,
            "predicted_delta_h": pred,
            "predicted_stage": issue + pred,
            "persistence_stage": issue,
        }))
    return pd.concat(frames, ignore_index=True).sort_values(
        ["issue_time", "valid_time", "horizon_min"]).reset_index(drop=True)


def frame_hash(frame):
    digest = hashlib.sha256()
    digest.update(json.dumps(list(frame.columns)).encode())
    digest.update(pd.util.hash_pandas_object(frame, index=True).to_numpy().tobytes())
    return digest.hexdigest()


def array_hash(*arrays):
    digest = hashlib.sha256()
    for array in arrays:
        array = np.asarray(array)
        digest.update(str(array.dtype).encode())
        digest.update(str(array.shape).encode())
        digest.update(array.tobytes())
    return digest.hexdigest()


def source_hash():
    # Script hash when run as .py; notebooks do not define __file__.
    source_path = globals().get("__file__")
    if source_path and Path(source_path).is_file():
        return hashlib.sha256(Path(source_path).read_bytes()).hexdigest()
    return hashlib.sha256(b"combined_tcn_lstm_colab_notebook_v1").hexdigest()


def settings_manifest():
    architecture = (
        {"channels": TCN_CHANNELS, "kernel_size": KERNEL_SIZE,
         "dilations": [1, 2, 4], "convolutions_per_block": 2, "head_hidden_units": 64}
        if ARCHITECTURE_LABEL == "TCN" else
        {"hidden_units": LSTM_HIDDEN_SIZE, "layers": LSTM_NUM_LAYERS,
         "head_hidden_units": 64})
    versions = {}
    for package in ["numpy", "pandas", "scikit-learn", "torch", "joblib"]:
        versions[package] = importlib.metadata.version(package)
    return {
        "schema_version": 1,
        "experiment": "fixed2025_causal_full5min",
        "description": "Forecasting downstream stage change using upstream hydrologic information and the antecedent downstream state.",
        "architecture": ARCHITECTURE_LABEL, "architecture_parameters": architecture,
        "training_years": TRAIN_YEARS, "validation_years": VAL_YEARS, "test_years": TEST_YEARS,
        "calendar_timezone": LOCAL_TZ, "timestamp_timezone": "UTC",
        "features": FEATURE_NAMES, "lookback_min": LOOKBACK_MIN,
        "lookback_steps": LOOKBACK_STEPS,
        "lookback_note": "36 samples t-175,...,t at a 5-minute timestep; nominal 180-minute lookback retained.",
        "horizons_min": HORIZONS_MIN, "timestep_min": TIMESTEP_MIN,
        "target": "delta_H(t,h) = H_US1(t+h) - H_US1(t), feet",
        "random_seed": RANDOM_SEED,
        "training_sample_stride": TRAIN_SAMPLE_STRIDE,
        "sampler": {"type": "WeightedRandomSampler", "replacement": True,
                    "draws_per_epoch": "number of eligible training sequences",
                    "weights": "8 if issue US1>=4 ft; else 4 if HW70 dStage_30>0.15 ft; else 1",
                    "all_eligible": True, "every_sample_each_epoch_guaranteed": False},
        "optimizer": "AdamW", "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "dropout": DROPOUT, "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS, "early_stopping_patience": PATIENCE,
        "early_stopping_min_delta": 1e-5,
        "early_stopping_monitor": "unweighted mean of validation batch composite losses (original behavior)",
        "gradient_clipping_max_norm": 1.0,
        "training_loss": {
            "equation": "0.5*r/(1+r) + 0.5*mean(direction_error)",
            "r": "mean_batch,horizon(abs(pred_delta-obs_delta))/max(training_5min_MASE_scale,1e-6)",
            "direction_error": "sigmoid(-sign(obs_delta)*pred_delta/0.05) if abs(obs_delta)>0.05; otherwise 1-exp(-(pred_delta/0.05)^2)",
            "deadband_ft": 0.05, "temperature_ft": 0.05},
        "MASE": {"numerator": "mean test abs(predicted_delta_h-observed_delta_h), separately per horizon",
                 "denominator": "mean abs(H_train[i+1]-H_train[i]) over finite adjacent 5-minute pairs in 2020-2023",
                 "horizon_dependent_denominator": False,
                 "one_means": "test forecast MAE equals the training 5-minute naive MAE; not equality to horizon-specific 2025 persistence"},
        "MDA": {"equation": "mean(sign(pred_delta)==sign(obs_delta))",
                "csv_units": "fraction in [0,1]", "figure_units": "percent",
                "zero": "correct only if predicted change is exactly zero",
                "near_zero": "strict positive/negative signs count, with no deadband"},
        "persistence": "predicted_delta=0; predicted_stage=H_US1(issue_time)",
        "preprocessing": {
            "alignment": "backward timestamp matching; source_time<=grid_time",
            "tolerance_seconds": 150, "interpolation": "none for predictors or targets",
            "missing_inputs": "StandardScaler fit only to train rows; missing standardized values become 0 (training mean); existing flags retained",
            "sequence_validity": "finite current HW70 and US1 stage, finite all seven future US1 labels, >=95% finite upstream stage in 36-step window",
            "boundary_policy": "feature history may use earlier observations; sequences and all labels stay within their assigned split",
            "availability_limit": "observation timestamp causality verified; historical cache does not establish telemetry publication latency or revision history"},
        "inference": "mean of 100 MC-dropout passes; original architecture-specific dropout activation",
        "mc_dropout_passes": MC_DROPOUT_PASSES,
        "device": str(DEVICE), "python": platform.python_version(),
        "platform": platform.platform(), "package_versions": versions,
        "script_sha256": source_hash(),
        "status": "configuration_only_not_trained",
    }


def prepare_data():
    raw, cache_hashes = load_causal_data()
    train_raw, val_raw, test_raw = split_by_calendar_year(raw)
    features = build_causal_features(raw)
    if list(features.columns) != FEATURE_NAMES:
        raise ValueError("Feature columns/order changed")
    train_f = features.loc[train_raw.index]
    scaler = StandardScaler().fit(train_f.to_numpy(dtype=np.float32))
    if not np.isfinite(scaler.mean_).all() or not np.isfinite(scaler.scale_).all():
        raise ValueError("At least one feature lacks finite training statistics")
    datasets = {}
    for split, data in [("train", train_raw), ("validation", val_raw), ("test", test_raw)]:
        ds = make_sequence_dataset(features.loc[data.index], data["us1_stage"], scaler,
                                   sample_stride=1)
        if len(ds) == 0:
            raise ValueError(f"No valid {split} sequences under causal missing-data policy")
        datasets[split] = ds
    if len(datasets["train"]) < 1000:
        raise ValueError("Fewer than 1000 valid training sequences; inspect coverage")
    scale = mase_scale_from_training(train_raw["us1_stage"])
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Training-only MASE scale is not finite and positive")
    manifest = settings_manifest()
    manifest.update({
        "status": "data_prepared_not_trained",
        "raw_cache_sha256": cache_hashes,
        "aligned_data_sha256": frame_hash(raw),
        "feature_data_sha256": frame_hash(features),
        "scaler_sha256": array_hash(scaler.mean_, scaler.scale_, scaler.var_),
        "scaler_mean": scaler.mean_.tolist(), "scaler_scale": scaler.scale_.tolist(),
        "mase_scale_training_ft": float(scale),
        "split_samples": {},
    })
    for split, ds in datasets.items():
        issues = ds.issue_times()
        manifest["split_samples"][split] = {
            "count": len(ds), "first_issue": str(issues[0]), "last_issue": str(issues[-1]),
            "issue_times_sha256": array_hash(issues.asi8),
            "dataset_sha256": array_hash(ds.X, ds.y, ds.valid),
        }
    weights = storm_sample_weights(datasets["train"])
    manifest["training_weight_counts"] = {
        str(int(w)): int(np.sum(weights == w)) for w in [1, 4, 8]}
    manifest["finite_stage_fraction_by_year"] = {
        str(year): raw.loc[local_year(raw.index) == year, ["hw70_stage", "us1_stage"]].notna().mean().to_dict()
        for year in range(2020, 2026)}
    return datasets, scaler, scale, manifest



# Original fixed-2025 comparison validation and aggregate plots (unchanged).
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
HORIZONS = [15, 30, 60, 90, 120, 150, 180]
KEY = ["issue_time", "valid_time", "horizon_min"]
VALUES = ["observed_stage", "observed_delta_h", "predicted_delta_h",
          "predicted_stage", "persistence_stage"]

def check_manifests(tcn, lstm):
    for name, manifest in [("TCN", tcn), ("LSTM", lstm)]:
        if manifest.get("architecture") != name or manifest.get("status") != "complete":
            raise ValueError(f"{name}: expected a completed matching-architecture run")
        required = {"experiment": "fixed2025_causal_full5min", "training_years": [2020, 2021, 2022, 2023],
                    "validation_years": [2024], "test_years": [2025], "horizons_min": HORIZONS,
                    "lookback_min": 180, "training_sample_stride": 1}
        for field, expected in required.items():
            if manifest.get(field) != expected:
                raise ValueError(f"{name}: invalid fixed experiment setting: {field}")
    common = ["schema_version", "experiment", "training_years", "validation_years", "test_years",
              "calendar_timezone", "timestamp_timezone", "features", "lookback_min", "lookback_steps",
              "horizons_min", "timestep_min", "target", "random_seed", "training_sample_stride",
              "sampler", "optimizer", "learning_rate", "weight_decay", "dropout", "batch_size",
              "max_epochs", "early_stopping_patience", "early_stopping_min_delta", "early_stopping_monitor",
              "gradient_clipping_max_norm", "training_loss", "MASE", "MDA", "persistence",
              "preprocessing", "inference", "mc_dropout_passes", "device", "python", "package_versions",
              "raw_cache_sha256", "aligned_data_sha256", "feature_data_sha256", "scaler_sha256",
              "scaler_mean", "scaler_scale", "mase_scale_training_ft", "split_samples", "training_weight_counts"]
    for field in common:
        if field not in tcn or field not in lstm or tcn[field] != lstm[field]:
            raise ValueError(f"Controlled-comparison manifest mismatch: {field}")
    scale = tcn["mase_scale_training_ft"]
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Invalid training-only MASE scale")


def read_run(folder):
    manifest = json.loads((folder / "run_manifest.json").read_text())
    path = folder / "predictions_2025.csv"
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != manifest.get("predictions_sha256"):
        raise ValueError(f"Prediction file hash mismatch: {path}")
    predictions = pd.read_csv(path, float_precision="round_trip")
    if len(predictions) != manifest.get("prediction_rows"):
        raise ValueError(f"Prediction row count differs from manifest: {folder}")
    if len(predictions) != manifest["split_samples"]["test"]["count"] * len(HORIZONS):
        raise ValueError(f"Incomplete test issue/horizon grid: {folder}")
    return predictions, manifest


def validate_predictions(frame, name):
    if frame.empty or any(col not in frame for col in KEY + VALUES):
        raise ValueError(f"{name}: empty predictions or missing required columns")
    frame = frame[KEY + VALUES].copy()
    for col in ["issue_time", "valid_time"]:
        frame[col] = pd.to_datetime(frame[col], utc=True, errors="raise")
        if frame[col].isna().any():
            raise ValueError(f"{name}: missing timestamp")
        if not (frame[col].dt.tz_convert("America/New_York").dt.year == 2025).all():
            raise ValueError(f"{name}: predictions outside the fixed 2025 test")
    if frame.duplicated(KEY).any():
        raise ValueError(f"{name}: duplicate (issue_time, valid_time, horizon_min) key")
    if sorted(frame["horizon_min"].unique().tolist()) != HORIZONS:
        raise ValueError(f"{name}: incorrect forecast horizons")
    if not ((frame["valid_time"]-frame["issue_time"]) == pd.to_timedelta(frame["horizon_min"], unit="min")).all():
        raise ValueError(f"{name}: valid time differs from issue time + horizon")
    if not frame.groupby("issue_time")["horizon_min"].nunique().eq(len(HORIZONS)).all():
        raise ValueError(f"{name}: not all seven horizons exist at every issue time")
    if not np.isfinite(frame[VALUES].to_numpy(dtype=float)).all():
        raise ValueError(f"{name}: nonfinite values; no silent sample removal allowed")
    for delta, stage in [("observed_delta_h", "observed_stage"), ("predicted_delta_h", "predicted_stage")]:
        if not np.allclose(frame["persistence_stage"]+frame[delta], frame[stage], rtol=0, atol=1e-6):
            raise ValueError(f"{name}: inconsistent delta/stage reconstruction")
    if not frame.groupby("issue_time")["persistence_stage"].nunique().eq(1).all():
        raise ValueError(f"{name}: persistence differs across horizons at the same issue time")
    return frame.sort_values(KEY).reset_index(drop=True)


def pair_predictions(tcn, lstm):
    tcn = validate_predictions(tcn, "TCN")
    lstm = validate_predictions(lstm, "LSTM")
    if len(tcn) != len(lstm) or not tcn[KEY].equals(lstm[KEY]):
        raise ValueError("TCN/LSTM issue times, valid times, horizons or sample counts differ")
    for field in ["observed_stage", "observed_delta_h", "persistence_stage"]:
        if not np.array_equal(tcn[field].to_numpy(), lstm[field].to_numpy()):
            raise ValueError(f"TCN/LSTM shared observation mismatch: {field}")
    # Equal, unique keys already verified. No inner join that can hide missing rows.
    paired = tcn[KEY + ["observed_stage", "observed_delta_h"]].copy()
    for name, frame in [("TCN", tcn), ("LSTM", lstm)]:
        paired[f"{name}_predicted_delta_h"] = frame["predicted_delta_h"]
        paired[f"{name}_predicted_stage"] = frame["predicted_stage"]
    paired["persistence_stage"] = tcn["persistence_stage"]
    return paired[[*KEY, "observed_stage", "observed_delta_h",
                   "TCN_predicted_delta_h", "LSTM_predicted_delta_h",
                   "TCN_predicted_stage", "LSTM_predicted_stage", "persistence_stage"]]


# MASE / MDA

def combined_metrics(paired, scale):
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("MASE requires a finite positive historical scale")
    rows = []
    for horizon, group in paired.groupby("horizon_min", sort=True):
        obs = group["observed_delta_h"].to_numpy(float)
        row = {"horizon_min": int(horizon)}
        for name in ["TCN", "LSTM"]:
            pred = group[f"{name}_predicted_delta_h"].to_numpy(float)
            row[f"{name}_MASE"] = float(np.mean(np.abs(pred-obs)) / scale)
            row[f"{name}_MDA"] = float(np.mean(np.sign(pred) == np.sign(obs)))
        rows.append(row)
    return pd.DataFrame(rows)[["horizon_min", "TCN_MASE", "LSTM_MASE", "TCN_MDA", "LSTM_MDA"]]


# OUTPUT — two aggregate figures; poster module adds event illustrations and design.

def make_figures(metrics, output):
    for metric, label, title, filename, factor in [
        ("MASE", "MASE (dimensionless)", "Fixed 2025 test: scaled stage-change forecast error", "figure1_MASE.png", 1),
        ("MDA", "Mean Directional Accuracy (%)", "Fixed 2025 test: downstream stage-change direction", "figure2_MDA.png", 100),
    ]:
        fig, ax = plt.subplots(figsize=(8.2, 5.0))
        for name, marker in [("TCN", "o"), ("LSTM", "s")]:
            ax.plot(metrics["horizon_min"], factor*metrics[f"{name}_{metric}"], marker=marker, label=name,
                    linestyle="-" if name == "TCN" else "--",
                    markerfacecolor="none" if name == "LSTM" else None)
        if metric == "MASE":
            ax.axhline(1, color="0.4", linestyle="--", label="Training 5-minute naive error reference")
            fig.text(0.5, 0.01, "MASE = 1: test forecast MAE equals training 5-minute naive MAE; not test persistence parity.",
                     ha="center", fontsize=8)
        else:
            ax.set_ylim(0, 100)
        ax.set(xlabel="Forecast horizon (minutes)", ylabel=label, title=title)
        ax.set_xticks(HORIZONS)
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout(rect=(0, 0.045 if metric == "MASE" else 0, 1, 1))
        fig.savefig(output / filename, dpi=300)
        plt.close(fig)



# Self-contained replacement for the unavailable POSTER_VISUALS_FIXED2025 module.
def draw_study_design(output):
    output.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.axis("off")
    boxes = [
        (0.02, "USGS cache\n2020–2025"),
        (0.24, "Causal 25 features\n180-min input"),
        (0.47, "TCN & LSTM\nshared split/loss"),
        (0.71, "2025 predictions\n15–180 minutes"),
    ]
    for x, text in boxes:
        ax.text(x, 0.5, text, transform=ax.transAxes, va="center", ha="left", fontsize=11,
                bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "edgecolor": "#666666"})
    for a,b in [(0.18,0.23),(0.405,0.46),(0.655,0.70)]:
        ax.annotate("", xy=(b,0.5), xytext=(a,0.5), xycoords="axes fraction",
                    arrowprops={"arrowstyle":"->", "lw":1.5})
    fig.tight_layout()
    fig.savefig(output / "study_design.png", dpi=200)
    plt.close(fig)


def make_event_figures(paired, output):
    """Plot available 2025 event windows; these are newly authored and not the missing poster module's figures."""
    output.mkdir(parents=True, exist_ok=True)
    valid_t = pd.to_datetime(paired["valid_time"], utc=True)
    local_t = valid_t.dt.tz_convert(LOCAL_TZ)
    for day in ["2025-08-11", "2025-08-18"]:
        start = pd.Timestamp(day, tz=LOCAL_TZ)
        keep = (local_t >= start) & (local_t < start + pd.Timedelta(days=1))
        if not keep.any():
            print(f"[PLOTS] No valid 2025 test samples for {day}; skipping this window")
            continue
        for h in (30, 60, 180):
            subset = paired.loc[keep & (paired.horizon_min == h)].copy().sort_values("valid_time")
            if subset.empty:
                continue
            t = pd.to_datetime(subset["valid_time"], utc=True).dt.tz_convert(LOCAL_TZ)
            fig, ax = plt.subplots(figsize=(11,4.3))
            for name, col, style in [
                ("Observed", "observed_stage", "-"),
                ("TCN", "TCN_predicted_stage", "-"),
                ("LSTM", "LSTM_predicted_stage", "--"),
                ("Persistence", "persistence_stage", ":"),
            ]:
                ax.plot(t, subset[col], style, lw=1.3, label=name)
            ax.set(xlabel="Valid time (America/New_York)", ylabel="US-1 stage (ft)",
                   title=f"2025-08-{day[-2:]} | {h}-min lead")
            ax.grid(alpha=0.25)
            ax.legend(ncol=4)
            fig.autofmt_xdate()
            fig.tight_layout()
            fig.savefig(output / f"event_{day}_{h}min_stage.png", dpi=180)
            plt.close(fig)


def run_comparison(output_root):
    output_root = Path(output_root).expanduser().resolve()
    tcn_dir = output_root / "TCN_fixed2025"
    lstm_dir = output_root / "LSTM_fixed2025"
    tcn, mt = read_run(tcn_dir)
    lstm, ml = read_run(lstm_dir)
    check_manifests(mt, ml)
    paired = pair_predictions(tcn, lstm)
    metrics = combined_metrics(paired, mt["mase_scale_training_ft"])
    output = output_root / "comparison_fixed2025"
    output.mkdir(parents=True, exist_ok=True)
    paired.to_csv(output / "combined_predictions_2025.csv", index=False)
    metrics.to_csv(output / "combined_metrics_2025.csv", index=False)
    combined_manifest = {
        "experiment": "fixed2025_causal_full5min", "paired_samples_verified": True,
        "paired_rows": len(paired), "unique_issue_times": paired["issue_time"].nunique(),
        "comparison_script_sha256": source_hash(),
        "units": {"stage_and_delta": "ft", "MDA_CSV": "fraction", "MDA_figure": "percent"},
        "runs": {"TCN": mt, "LSTM": ml},
        "source_run_directories": {"TCN": str(tcn_dir), "LSTM": str(lstm_dir)},
    }
    (output / "run_manifest.json").write_text(json.dumps(combined_manifest, indent=2)+"\n")
    make_figures(metrics, output)
    poster_dir = output / "poster_figures"
    draw_study_design(poster_dir)
    make_event_figures(paired, poster_dir)
    print(metrics.to_string(index=False))
    print(f"Verified {len(paired):,} paired rows. Comparison outputs: {output}")
    return metrics


CACHE_SERIES = (
    (HW70_SITE, PRECIP_CD, "hw70_precip"),
    (HW70_SITE, STAGE_CD, "hw70_stage"),
    (US1_SITE, STAGE_CD, "us1_stage"),
)


def missing_cache_specs(cache_dir):
    """Return only the missing annual site/parameter/column/year combinations."""
    folder = Path(cache_dir).expanduser()
    return [(year, site, code, name)
            for year in range(START_YEAR, END_YEAR + 1)
            for site, code, name in CACHE_SERIES
            if not (folder / f"{year}_{site}_{code}_{name}.csv").is_file()]


def validate_cache(cache_dir):
    """Check all required files are present and have the expected minimal schema."""
    folder = Path(cache_dir).expanduser()
    missing = missing_cache_specs(folder)
    if missing:
        filenames = [f"{year}_{site}_{code}_{name}.csv"
                     for year, site, code, name in missing]
        raise FileNotFoundError(
            f"Missing {len(filenames)} of 18 expected observation cache files in {folder}:\n" +
            "\n".join(filenames) +
            "\nEnable download_missing=True (Colab: DOWNLOAD_MISSING_CACHE=True) "
            "to fetch only the missing files."
        )
    for year in range(START_YEAR, END_YEAR + 1):
        for site, code, name in CACHE_SERIES:
            file = folder / f"{year}_{site}_{code}_{name}.csv"
            if file.stat().st_size == 0:
                raise ValueError(f"Empty cache: {file}. Remove it to trigger a fresh download.")
            try:
                head = pd.read_csv(file, nrows=2)
            except Exception as exc:
                raise ValueError(f"Unreadable cache: {file}. Remove it to re-download.") from exc
            if head.empty:
                raise ValueError(f"Cache has no data rows: {file}. Remove it to re-download.")
            if "datetime" not in head.columns and "index" not in head.columns:
                raise ValueError(f"Cache lacks datetime column: {file}. Remove it to re-download.")
            if name not in head.columns:
                raise ValueError(f"Cache lacks '{name}' column: {file}. Remove it to re-download.")
    print(f"Validated all 18 annual observation caches in {folder}")


def ensure_cache(cache_dir, download_missing=True):
    """Reuse existing CSVs; fetch only missing raw USGS annual observations.

    NWIS site/parameter series: HW70 rainfall, HW70 stage and US1 stage.
    Fail closed: never train after failed or partial downloads.
    """
    global CACHE_DIR, ALLOW_DOWNLOAD
    CACHE_DIR = Path(cache_dir).expanduser().resolve()
    missing = missing_cache_specs(CACHE_DIR)
    if missing and not download_missing:
        validate_cache(CACHE_DIR)  # Raises with the precise missing-file names.
    if missing:
        if not CACHE_DIR.parent.exists():
            raise FileNotFoundError(
                f"Cache parent path is not mounted/available: {CACHE_DIR.parent}. "
                "In Colab, mount Google Drive before running the experiment."
            )
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        ALLOW_DOWNLOAD = True
        print(f"[USGS] {len(missing)} missing of 18 annual CSVs; downloading only missing files.")
        print("[USGS] Download coverage is fixed to 2020–2025 (2025 remains the holdout test year).")
        for year, site, code, name in missing:
            download_year_series(site, code, name, year)
    else:
        print("[USGS] Complete cache found; no network requests required.")
    validate_cache(CACHE_DIR)
    return CACHE_DIR


def run_one_architecture(architecture, datasets, scaler, scale, base_manifest, output_root):
    global ARCHITECTURE_LABEL, OUTPUT_DIR
    ARCHITECTURE_LABEL = architecture
    OUTPUT_DIR = Path(output_root).expanduser().resolve() / f"{architecture}_fixed2025"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest_file = OUTPUT_DIR / "run_manifest.json"
    predictions_file = OUTPUT_DIR / "predictions_2025.csv"
    if manifest_file.exists() or predictions_file.exists():
        if not (manifest_file.exists() and predictions_file.exists()):
            raise FileExistsError(f"Partial/incomplete {architecture} run exists in {OUTPUT_DIR}. "
                                  "Choose a clean output directory to retrain; will not overwrite.")
        # Explicitly verify this run matches the current prepared cache/design.
        prior = json.loads(manifest_file.read_text())
        config = settings_manifest()
        current = {**base_manifest, **config}
        fields = ["architecture", "raw_cache_sha256", "aligned_data_sha256",
                  "feature_data_sha256", "scaler_sha256", "split_samples",
                  "training_weight_counts", "mase_scale_training_ft", "random_seed",
                  "batch_size", "mc_dropout_passes", "training_loss", "architecture_parameters"]
        if any(prior.get(field) != current.get(field) for field in fields):
            raise ValueError(f"Existing {architecture} run differs from current cache/settings. "
                             f"Use a new OUTPUT_ROOT. Folder: {OUTPUT_DIR}")
        read_run(OUTPUT_DIR)
        print(f"[REUSE] Verified completed {architecture} artifacts in {OUTPUT_DIR}")
        return
    manifest = {**base_manifest, **settings_manifest()}
    manifest["status"] = "training"
    manifest_file.write_text(json.dumps(manifest, indent=2)+"\n")
    joblib.dump(scaler, OUTPUT_DIR / "feature_scaler.joblib")
    try:
        model, history = train_model(
            datasets["train"], datasets["validation"],
            n_features=len(FEATURE_NAMES), mase_scale=scale)
        mc_out = predict_dataset_mc(model, datasets["test"])
        predictions = prediction_table(mc_out)
        primary_metrics(predictions, scale)
        write_output_csv(predictions, "predictions_2025.csv")
        manifest["status"] = "complete"
        manifest["epochs_completed"] = len(history)
        manifest["trained_parameters"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
        manifest["prediction_rows"] = len(predictions)
        manifest["predictions_sha256"] = hashlib.sha256(predictions_file.read_bytes()).hexdigest()
        manifest_file.write_text(json.dumps(manifest, indent=2)+"\n")
        print(f"[{architecture}] Finished: {OUTPUT_DIR}")
    finally:
        if "model" in locals():
            del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


def run_pipeline(mode="all", cache_dir=None, output_root=None, download_missing=True):
    """'prepare' audits cached observations, 'all' trains both models and compares; 'compare' uses completed runs."""
    global CACHE_DIR, OUTPUT_DIR, ARCHITECTURE_LABEL, ALLOW_DOWNLOAD
    if mode not in {"prepare", "all", "compare"}:
        raise ValueError("mode must be prepare, all, or compare")
    ALLOW_DOWNLOAD = bool(download_missing)
    output_root = Path(output_root or (ROOT / "crabtree_colab_results")).expanduser().resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    if mode == "compare":
        return run_comparison(output_root)
    CACHE_DIR = Path(cache_dir or (ROOT / "shared_usgs_cache")).expanduser().resolve()
    ensure_cache(CACHE_DIR, download_missing=download_missing)
    ARCHITECTURE_LABEL = "TCN"
    OUTPUT_DIR = output_root / "TCN_fixed2025"
    print(f"Compute device: {DEVICE} | batch size: {BATCH_SIZE}")
    datasets, scaler, scale, manifest = prepare_data()  # one preparation shared by BOTH models
    print("Samples:", {name: len(ds) for name, ds in datasets.items()})
    print("Training-only 5-min naive MAE / MASE denominator (ft):", scale)
    if mode == "prepare":
        audit_path = output_root / "data_audit.json"
        audit_path.write_text(json.dumps(manifest, indent=2)+"\n")
        print(f"Data audit saved: {audit_path}")
        return manifest
    for architecture in ["TCN", "LSTM"]:
        run_one_architecture(architecture, datasets, scaler, scale, manifest, output_root)
    return run_comparison(output_root)



# ────────────────────────────────────────────────────────────────────────────
# LOSS-FUNCTION SENSITIVITY STUDY AND FEATURE IMPORTANCE
# Compare four objectives with identical data, architectures, optimizer and seed.
# ────────────────────────────────────────────────────────────────────────────
FEATURE_LOSS_NAME = "composite"     # Preselected for feature importance (not selected by test accuracy).
SHAP_TEST_SAMPLES = 256             # Uniform 2025 issue times to explain.
SHAP_BACKGROUND_SAMPLES = 64        # Training-window background; never use test data as background.
SHAP_NSAMPLES = 64                  # Expected-gradient Monte Carlo samples per explained input.
SHAP_CHUNK_SIZE = 16                # Limit memory use, especially on Colab GPUs.
SHAP_EXPLAINER_BATCH_SIZE = 16      # Internal SHAP expected-gradient batch size.
FEATURE_GROUPS = {
    "HW70 stage level/history": ["stage", "stage_lag30", "stage_lag60", "stage_roll30_mean", "stage_roll60_max"],
    "HW70 stage changes": ["dStage_5", "dStage_15", "dStage_30"],
    "US1 current stage": ["us1_stage"],
    "US1 stage changes": ["us1_dStage_5", "us1_dStage_15", "us1_dStage_30"],
    "Rainfall/accumulation": ["rain", "rain_sum_15", "rain_sum_30", "rain_sum_60", "rain_sum_120", "rain_sum_180"],
    "Observation availability": ["rain_available", "us1_available", "rain_cov_15", "rain_cov_30", "rain_cov_60", "rain_cov_120", "rain_cov_180"],
}
assert sorted(sum(FEATURE_GROUPS.values(), [])) == sorted(FEATURE_NAMES), "Feature groups must cover exactly 25 features"


def make_architecture(architecture):
    if architecture == "TCN":
        return HydroTCN(n_features=len(FEATURE_NAMES), channels=TCN_CHANNELS,
                        kernel_size=KERNEL_SIZE, dropout=DROPOUT, n_outputs=len(HORIZONS_MIN)).to(DEVICE)
    if architecture == "LSTM":
        return HydroLSTM(n_features=len(FEATURE_NAMES), hidden_size=LSTM_HIDDEN_SIZE,
                         num_layers=LSTM_NUM_LAYERS, dropout=DROPOUT,
                         n_outputs=len(HORIZONS_MIN)).to(DEVICE)
    raise ValueError(f"Unknown architecture: {architecture}")


def loss_run_manifest(base_manifest, architecture, objective, ref):
    global ARCHITECTURE_LABEL
    ARCHITECTURE_LABEL = architecture
    m = {**base_manifest, **settings_manifest()}
    m["loss_name"] = objective
    m["training_loss"] = {
        "loss_name": objective,
        "equation": TRAINING_LOSS_DESCRIPTIONS[objective],
        "stage_or_delta": "KGE and NSE normalization use absolute stage; RMSE and composite use delta (same residuals)",
        "reference": ("train-only full-split variance denominator, horizon-specific" if objective == "NSE" else
                      "batch sample mean/std/correlation with variance regularization" if objective == "KGE" else
                      "original composite loss from supplied scripts" if objective == "composite" else "batch RMSE"),
        "kge_version": "Gupta et al 2009: r, alpha=std_ratio, beta=mean_ratio",
        "nse_test_note": "Reported conventional NSE uses 2025 observed-stage variance; training NSE uses fixed training-stage variance",
    }
    m["train_loss_reference"] = ref
    m["early_stopping_monitor"] = f"unweighted mean of validation batch {objective} objective (same early-stopping schedule)"
    m["feature_importance_design"] = "SHAP GradientExplainer on composite model; train-window background, 2025 explanation windows, deterministic dropout-off inference"
    return m


def run_architecture_loss(architecture, objective, datasets, scaler, scale, base_manifest, ref, root):
    global ARCHITECTURE_LABEL, OUTPUT_DIR
    if objective not in LOSS_NAMES:
        raise ValueError(objective)
    ARCHITECTURE_LABEL = architecture
    objective_root = Path(root) / "loss_experiments" / objective
    OUTPUT_DIR = objective_root / f"{architecture}_fixed2025"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = OUTPUT_DIR / "run_manifest.json"
    pred_path = OUTPUT_DIR / "predictions_2025.csv"
    expected = loss_run_manifest(base_manifest, architecture, objective, ref)
    if manifest_path.exists() or pred_path.exists():
        if not manifest_path.exists() or not pred_path.exists():
            raise RuntimeError(f"Incomplete run for {architecture}/{objective}: {OUTPUT_DIR}. "
                               "Do not mix runs. Rename this folder before retrying.")
        observed = json.loads(manifest_path.read_text())
        keys = ("architecture", "loss_name", "training_loss", "train_loss_reference", "architecture_parameters",
                "raw_cache_sha256", "aligned_data_sha256", "feature_data_sha256", "scaler_sha256",
                "split_samples", "mase_scale_training_ft", "random_seed", "batch_size", "mc_dropout_passes",
                "training_weight_counts", "max_epochs", "early_stopping_patience", "optimizer", "sampler")
        differing = [k for k in keys if observed.get(k) != expected.get(k)]
        if differing:
            raise ValueError(f"Cannot reuse {architecture}/{objective}: different design {differing}. "
                             f"Use new output root, not the prior run: {OUTPUT_DIR}")
        read_run(OUTPUT_DIR)  # SHA256 and expected row count verified.
        print(f"[REUSE] Verified completed {architecture}/{objective} run")
        return OUTPUT_DIR
    manifest = dict(expected, status="training")
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
    joblib.dump(scaler, OUTPUT_DIR / "feature_scaler.joblib")
    model = None
    try:
        model, history = train_model(datasets["train"], datasets["validation"],
                                     len(FEATURE_NAMES), scale, loss_name=objective,
                                     train_reference=ref)
        # Reset inference random streams identically across runs; retain 100 MC passes.
        set_global_seed(RANDOM_SEED + 10000)
        mc = predict_dataset_mc(model, datasets["test"], n_passes=MC_DROPOUT_PASSES)
        predictions = prediction_table(mc)
        primary_metrics(predictions, scale)
        write_output_csv(predictions, "predictions_2025.csv")
        manifest.update(status="complete", epochs_completed=len(history),
                        trained_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                        prediction_rows=len(predictions),
                        predictions_sha256=hashlib.sha256(pred_path.read_bytes()).hexdigest(),
                        checkpoint_name=f"hydro_{architecture.lower()}_delta_mc.pt")
        manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
        print(f"[COMPLETE] {objective} / {architecture}: {len(predictions):,} prediction rows")
        return OUTPUT_DIR
    finally:
        if model is not None:
            del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


def _pearson_kge(obs, pred):
    """Conventional evaluation KGE2009 on absolute stage. Undefined -> NaN."""
    o = np.asarray(obs, dtype=np.float64)
    p = np.asarray(pred, dtype=np.float64)
    so, sp = np.std(o), np.std(p)
    if len(o) < 3 or so < 1e-10 or sp < 1e-10 or abs(np.mean(o)) < 1e-10:
        return (np.nan, np.nan)
    r = float(np.corrcoef(o, p)[0, 1])
    alpha, beta = sp / so, np.mean(p) / np.mean(o)
    kge = 1.0 - np.sqrt((r - 1.0)**2 + (alpha - 1.0)**2 + (beta - 1.0)**2)
    return float(kge), r


def metric_table_from_predictions(predictions, scale, arch, objective):
    validated = validate_predictions(predictions, f"{arch}/{objective}")
    rows = []
    for horizon, group in validated.groupby("horizon_min", sort=True):
        obs = group["observed_stage"].to_numpy(float)
        pred = group["predicted_stage"].to_numpy(float)
        naive = group["persistence_stage"].to_numpy(float)
        od = group["observed_delta_h"].to_numpy(float)
        pd_ = group["predicted_delta_h"].to_numpy(float)
        err = pred - obs
        mae = float(np.mean(np.abs(err)))
        pst_mae = float(np.mean(np.abs(obs - naive)))
        var_sum = float(np.sum((obs - np.mean(obs))**2))
        nse = float(1.0 - np.sum(err**2) / var_sum) if var_sum > 1e-10 else np.nan
        kge, r = _pearson_kge(obs, pred)
        true_class = np.where(od > DIRECTION_DEADBAND_FT, 1, np.where(od < -DIRECTION_DEADBAND_FT, -1, 0))
        pred_class = np.where(pd_ > DIRECTION_DEADBAND_FT, 1, np.where(pd_ < -DIRECTION_DEADBAND_FT, -1, 0))
        high = obs >= HIGH_STAGE_FT
        rows.append({
            "architecture": arch, "training_loss": objective, "horizon_min": int(horizon),
            "n_test_issues": len(group), "RMSE_ft": float(np.sqrt(np.mean(err**2))),
            "MAE_ft": mae, "MASE": float(mae / scale),
            "MDA_raw": float(np.mean(np.sign(od) == np.sign(pd_))),
            "MDA_deadband": float(np.mean(true_class == pred_class)),
            "NSE_stage": nse, "KGE_stage": kge, "Pearson_r_stage": r,
            "PBIAS_stage_pct": float(100.0*np.sum(err)/np.sum(obs)) if abs(np.sum(obs))>1e-10 else np.nan,
            "persistence_MAE_ft": pst_mae,
            "skill_vs_persistence": float(1-mae/pst_mae) if pst_mae>1e-10 else np.nan,
            "high_stage_n": int(high.sum()),
            "high_stage_RMSE_ft": float(np.sqrt(np.mean(err[high]**2))) if high.any() else np.nan,
        })
    return pd.DataFrame(rows)


def collect_study_metrics(root, require_complete=True):
    """Always verify file hashes and matching issue-time rows before consolidation."""
    base = Path(root) / "loss_experiments"
    rows=[]
    reference_keys=None
    for objective in LOSS_NAMES:
        location = base / objective
        if not ((location / "TCN_fixed2025/run_manifest.json").exists() and
                (location / "LSTM_fixed2025/run_manifest.json").exists()):
            if require_complete:
                raise FileNotFoundError(f"Missing completed runs for {objective}: {location}")
            return None
        tcn, mt = read_run(location / "TCN_fixed2025")
        lstm, ml = read_run(location / "LSTM_fixed2025")
        check_manifests(mt, ml)
        if mt.get("loss_name") != objective or ml.get("loss_name") != objective:
            raise ValueError(f"Loss mismatch in {objective} folder")
        common = {k:mt[k] for k in ("raw_cache_sha256", "aligned_data_sha256", "feature_data_sha256",
                                    "scaler_sha256", "split_samples", "mase_scale_training_ft", "random_seed")}
        if reference_keys is None:
            reference_keys = common
        elif reference_keys != common:
            raise ValueError(f"Across-loss dataset/split/seed mismatch for {objective}")
        pair_predictions(tcn, lstm)
        for arch, pred in (("TCN",tcn),("LSTM",lstm)):
            rows.append(metric_table_from_predictions(pred, mt["mase_scale_training_ft"], arch, objective))
    table = pd.concat(rows,ignore_index=True)
    if len(table) != len(LOSS_NAMES)*2*len(HORIZONS_MIN):
        raise ValueError("Unexpected loss-sensitivity table shape")
    return table


def plot_loss_study(table, output):
    output=Path(output); output.mkdir(parents=True,exist_ok=True)
    table.to_csv(output / "loss_metrics_by_horizon.csv",index=False)
    columns=["RMSE_ft", "MAE_ft", "MASE", "MDA_raw", "MDA_deadband", "NSE_stage", "KGE_stage",
             "skill_vs_persistence", "high_stage_RMSE_ft", "PBIAS_stage_pct"]
    summary = table.groupby(["architecture", "training_loss"],as_index=False)[columns].mean(numeric_only=True)
    summary.to_csv(output / "loss_sensitivity_mean_across_7_horizons.csv",index=False)
    changes=[]
    for arch in ("TCN","LSTM"):
        base=table.loc[(table.architecture==arch)&(table.training_loss=="composite")].set_index("horizon_min")
        for obj in LOSS_NAMES:
            sub=table.loc[(table.architecture==arch)&(table.training_loss==obj)].set_index("horizon_min")
            for h in HORIZONS_MIN:
                b,s=base.loc[h],sub.loc[h]
                change={"architecture":arch,"training_loss":obj,"horizon_min":h}
                for metric in ("RMSE_ft", "MASE", "MAE_ft"):
                    denom=abs(b[metric]); change[metric+"_change_pct"]=100.0*(s[metric]-b[metric])/denom if denom>1e-10 else np.nan
                for metric in ("MDA_raw", "MDA_deadband", "skill_vs_persistence"):
                    change[metric+"_change_pp"]=100.0*(s[metric]-b[metric])
                for metric in ("NSE_stage", "KGE_stage", "high_stage_RMSE_ft"):
                    change[metric+"_change_absolute"]=s[metric]-b[metric]
                changes.append(change)
    change_df=pd.DataFrame(changes)
    change_df.to_csv(output / "loss_sensitivity_changes_vs_composite_by_horizon.csv",index=False)
    means=change_df.groupby(["architecture","training_loss"],as_index=False).mean(numeric_only=True).drop(columns=["horizon_min"])
    means.to_csv(output / "loss_sensitivity_changes_vs_composite_summary.csv",index=False)
    metrics=[("RMSE_ft", "RMSE (ft)"), ("MASE","MASE"),("MDA_raw","Raw MDA (fraction)"),
             ("NSE_stage","NSE on stage"),("KGE_stage","KGE2009 on stage"),
             ("skill_vs_persistence","MAE skill vs 2025 persistence")]
    for arch in ("TCN","LSTM"):
        for key,label in metrics:
            fig,ax=plt.subplots(figsize=(8.3,4.8))
            for name in LOSS_NAMES:
                sub=table[(table.architecture==arch)&(table.training_loss==name)].sort_values("horizon_min")
                ax.plot(sub.horizon_min,sub[key],marker="o",label=name)
            ax.set(xlabel="Forecast horizon (min)",ylabel=label,title=f"{arch}: evaluation of four trained objectives on 2025")
            ax.set_xticks(HORIZONS_MIN);ax.grid(alpha=0.25);ax.legend(title="Training loss",fontsize=8)
            fig.tight_layout();fig.savefig(output/f"{arch}_{key}_across_horizons.png",dpi=160);plt.close(fig)
        # Relative-to-composite sensitivity heatmap. Each cell = 7-horizon-average relative benefit.
        # Errors: positive means improvement (reduction). Skill/NSE/KGE/MDA: positive means improvement.
        cols=[("RMSE_ft_change_pct","RMSE benefit (%)",-1),
              ("MASE_change_pct","MASE benefit (%)",-1),
              ("MDA_raw_change_pp","MDA gain (pp)",1),
              ("NSE_stage_change_absolute","NSE gain",1),
              ("KGE_stage_change_absolute","KGE gain",1),
              ("skill_vs_persistence_change_pp","Skill gain (pp)",1)]
        subset=means[(means.architecture==arch)].set_index("training_loss").loc[list(LOSS_NAMES)]
        vals=np.array([[(subset.loc[obj,col] * sign) for col,_,sign in cols] for obj in LOSS_NAMES],dtype=float)
        finite=np.abs(vals[np.isfinite(vals)])
        lim=max(1.0,float(np.percentile(finite,90))) if len(finite) else 1.0
        fig,ax=plt.subplots(figsize=(12,4.5))
        img=ax.imshow(vals,aspect="auto",cmap="RdBu",vmin=-lim,vmax=lim)
        ax.set_xticks(range(len(cols)),[c[1] for c in cols],rotation=25,ha="right")
        ax.set_yticks(range(len(LOSS_NAMES)),list(LOSS_NAMES))
        ax.set_title(f"{arch}: mean improvement vs original composite across 7 horizons")
        for i in range(vals.shape[0]):
            for j in range(vals.shape[1]):
                ax.text(j,i,f"{vals[i,j]:+.2f}" if np.isfinite(vals[i,j]) else "N/A",ha="center",va="center",fontsize=8,
                        color="black" if abs(vals[i,j])<lim*.6 else "white")
        fig.colorbar(img,ax=ax,label="Red = positive benefit; blue = deterioration (mixed units: read labels)")
        fig.tight_layout();fig.savefig(output/f"{arch}_loss_sensitivity_heatmap.png",dpi=170);plt.close(fig)
    return summary, change_df


def _sample_windows_for_shap(ds, n, seed):
    """Uniformly select complete causal windows without modifying any predictor.

    The caller supplies the TRAIN split to build a background distribution and
    the TEST split to select windows for explanation. The valid-index sampler
    is seeded, stable, and independent of model results.
    """
    if n < 1 or len(ds.valid) == 0:
        raise ValueError("SHAP requires positive sample counts and nonempty valid windows")
    rng = np.random.default_rng(seed)
    chosen = np.sort(rng.choice(len(ds.valid), size=min(int(n), len(ds.valid)), replace=False))
    rows = ds.valid[chosen]
    X = np.stack([ds.X[i - ds.lookback + 1:i + 1] for i in rows]).astype(np.float32)
    if X.shape[1:] != (LOOKBACK_STEPS, len(FEATURE_NAMES)) or not np.isfinite(X).all():
        raise ValueError(f"Invalid SHAP window dimensions/values: {X.shape}")
    return X, pd.DatetimeIndex(ds.index[rows])


def _shap_config(arch, objective, checkpoint, datasets, n_samples, background_samples, nsamples):
    """Cache fingerprint; validates the model and the underlying split data."""
    import shap
    return {
        "method": "shap.GradientExplainer expected gradients, multi-output",
        "version": 3,
        "shap_version": shap.__version__,
        "architecture": arch, "training_loss": objective,
        "checkpoint_sha256": hashlib.sha256(Path(checkpoint).read_bytes()).hexdigest(),
        "n_explain_requested": int(n_samples),
        "n_background_requested": int(background_samples),
        "expected_gradient_nsamples": int(nsamples),
        "sample_seed": int(RANDOM_SEED + 613),
        "background_seed": int(RANDOM_SEED + 617),
        "horizons_min": HORIZONS_MIN,
        "feature_names": FEATURE_NAMES,
        "lookback_steps": LOOKBACK_STEPS,
        "test_issue_count": len(datasets["test"]),
        "train_issue_count": len(datasets["train"]),
        "train_data_sha256": array_hash(datasets["train"].X, datasets["train"].valid),
        "test_data_sha256": array_hash(datasets["test"].X, datasets["test"].valid),
    }


def shap_gradient_attributions(model, train_ds, test_ds,
                               n_samples=SHAP_TEST_SAMPLES,
                               background_samples=SHAP_BACKGROUND_SAMPLES,
                               nsamples=SHAP_NSAMPLES):
    """Explain 7 forecast ΔH outputs with SHAP expected gradients.

    Input: standardized [sample, 36 timestamps, 25 features] causal history.
    Output: SHAP [sample, 36 timestamps, 25 features, 7 horizons], in feet.
    The explainer background comes ONLY from 2020–23 training windows; explained
    windows come ONLY from the 2025 holdout. Dropout and MC sampling are OFF.

    CuDNN is temporarily disabled for gradients through PyTorch LSTM in eval
    mode; normal GPU training and inference settings remain unchanged.
    """
    import shap
    if n_samples < 1 or background_samples < 2 or nsamples < 2:
        raise ValueError("SHAP samples must be >=1 and background/nsamples must be >=2")
    background, background_times = _sample_windows_for_shap(train_ds, background_samples, RANDOM_SEED + 617)
    X, issue_times = _sample_windows_for_shap(test_ds, n_samples, RANDOM_SEED + 613)
    model.eval()
    bg_tensor = torch.from_numpy(background).to(DEVICE)
    vals = []
    with torch.backends.cudnn.flags(enabled=False):
        explainer = shap.GradientExplainer(model, bg_tensor, batch_size=SHAP_EXPLAINER_BATCH_SIZE)
        for j in range(0, len(X), SHAP_CHUNK_SIZE):
            batch = torch.from_numpy(X[j:j + SHAP_CHUNK_SIZE]).to(DEVICE)
            # SHAP >0.45 returns [batch, time, predictor, output] for multi-output PyTorch.
            sv = explainer.shap_values(batch, nsamples=int(nsamples), rseed=int(RANDOM_SEED + 911 + j))
            if isinstance(sv, list):
                if len(sv) != len(HORIZONS_MIN):
                    raise RuntimeError(f"SHAP returned {len(sv)} outputs; need {len(HORIZONS_MIN)}")
                sv = np.stack(sv, axis=-1)
            sv = np.asarray(sv)
            if sv.ndim == 5 and sv.shape[-2] == 1:
                sv = sv[..., 0, :]
            expected = (len(batch), LOOKBACK_STEPS, len(FEATURE_NAMES), len(HORIZONS_MIN))
            if sv.shape != expected:
                raise RuntimeError(f"SHAP output shape {sv.shape} does not match {expected}; check SHAP version")
            if not np.isfinite(sv).all():
                raise RuntimeError("Nonfinite SHAP values; cannot produce trustworthy rankings")
            vals.append(sv.astype(np.float32))
            print(f"  [SHAP] explained {min(j + len(batch), len(X))}/{len(X)} holdout windows")
    shap_values = np.concatenate(vals, axis=0)
    with torch.no_grad():
        background_output = model(bg_tensor).detach().cpu().numpy().mean(axis=0)
        predictions = np.concatenate([
            model(torch.from_numpy(X[j:j + SHAP_CHUNK_SIZE]).to(DEVICE)).detach().cpu().numpy()
            for j in range(0, len(X), SHAP_CHUNK_SIZE)
        ], axis=0)
    # Expected gradients approximate, not exact, attribution additivity.
    residual = predictions - background_output[None, :] - shap_values.sum(axis=(1,2))
    return {
        "shap_values": shap_values,
        "issue_time_utc": np.asarray(issue_times.strftime('%Y-%m-%dT%H:%M:%SZ'), dtype='U32'),
        "issue_feature_scaled": X[:, -1, :],
        "background_issue_time_utc": np.asarray(background_times.strftime('%Y-%m-%dT%H:%M:%SZ'), dtype='U32'),
        "background_mean_output_ft": background_output,
        "output_delta_h_ft": predictions,
        "expected_gradient_additivity_mae_ft": np.abs(residual).mean(axis=0),
    }


def shap_tables(arch, attribution):
    """Global mean(|summed history SHAP|), not a skill-degradation metric.

    Individual: sum SHAP over 36 timesteps for each predictor per issue time.
    Group: sum signed contributions of all members first, then take abs.
    This preserves possible cancellation within an interpretable group.
    """
    phi = attribution["shap_values"]
    window_phi = phi.sum(axis=1)  # sample × 25 predictor histories × 7 forecast horizons
    n = len(window_phi)
    feature_rows, group_rows, temporal_rows = [], [], []
    for j, h in enumerate(HORIZONS_MIN):
        for k, fname in enumerate(FEATURE_NAMES):
            p = window_phi[:,k,j]
            feature_rows.append({"architecture":arch, "training_loss":FEATURE_LOSS_NAME,
                                 "feature":fname, "horizon_min":h,
                                 "mean_abs_shap_ft":float(np.abs(p).mean()),
                                 "mean_signed_shap_ft":float(p.mean()),
                                 "n_explained_issues":n})
        for group, cols in FEATURE_GROUPS.items():
            positions=[FEATURE_NAMES.index(c) for c in cols]
            p=window_phi[:,positions,j].sum(axis=1)
            group_rows.append({"architecture":arch, "training_loss":FEATURE_LOSS_NAME,
                               "group":group, "horizon_min":h,
                               "mean_abs_shap_ft":float(np.abs(p).mean()),
                               "mean_signed_shap_ft":float(p.mean()),
                               "n_explained_issues":n})
        for time_i in range(LOOKBACK_STEPS):
            temporal_rows.append({"architecture":arch,"horizon_min":h,
                                  "minutes_before_issue":(LOOKBACK_STEPS-1-time_i)*TIMESTEP_MIN,
                                  "mean_abs_shap_ft":float(np.abs(phi[:,time_i,:,j]).sum(axis=1).mean())})
    return pd.DataFrame(feature_rows),pd.DataFrame(group_rows),pd.DataFrame(temporal_rows)


def plot_shap_results(feature_df,group_df,temporal_df, attribution_by_arch,output):
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    feature_df.to_csv(output/'shap_feature_importance_by_horizon.csv', index=False)
    group_df.to_csv(output/'shap_group_importance_by_horizon.csv', index=False)
    temporal_df.to_csv(output/'shap_temporal_importance.csv', index=False)
    ranked_features=(feature_df.groupby(['architecture','feature'],as_index=False)['mean_abs_shap_ft']
                     .mean().sort_values(['architecture','mean_abs_shap_ft'],ascending=[True,False]))
    ranked_features['rank']=ranked_features.groupby('architecture').cumcount()+1
    ranked_groups=(group_df.groupby(['architecture','group'],as_index=False)['mean_abs_shap_ft']
                   .mean().sort_values(['architecture','mean_abs_shap_ft'],ascending=[True,False]))
    ranked_groups['rank']=ranked_groups.groupby('architecture').cumcount()+1
    ranked_features.to_csv(output/'shap_feature_importance_ranked.csv',index=False)
    ranked_groups.to_csv(output/'shap_group_importance_ranked.csv',index=False)
    for arch in ('TCN','LSTM'):
        for dtype,ranked,title_col in [('feature',ranked_features,'feature'),('group',ranked_groups,'group')]:
            sub=ranked[ranked.architecture==arch].sort_values('mean_abs_shap_ft')
            fig,ax=plt.subplots(figsize=(9.8,9 if dtype=='feature' else 5.0))
            ax.barh(sub[title_col],sub.mean_abs_shap_ft)
            ax.set(xlabel='Mean |SHAP contribution to ΔH| (ft)',
                   title=f'{arch}: {dtype} SHAP ranking (composite loss, mean of seven horizons)')
            fig.tight_layout();fig.savefig(output/f'{arch}_shap_{dtype}_ranking.png',dpi=180);plt.close(fig)
        for df,idx,filename,title in [
            (feature_df,'feature',f'{arch}_shap_feature_by_horizon.png','Predictor SHAP by forecast horizon'),
            (group_df,'group',f'{arch}_shap_group_by_horizon.png','Grouped SHAP by forecast horizon'),
            (temporal_df,'minutes_before_issue',f'{arch}_shap_temporal_by_horizon.png','Time-lag SHAP by forecast horizon')]:
            sub=df[df.architecture==arch]
            hm=sub.pivot(index=idx,columns='horizon_min',values='mean_abs_shap_ft')
            if idx=='feature':hm=hm.reindex(FEATURE_NAMES)
            elif idx=='group':hm=hm.reindex(list(FEATURE_GROUPS))
            else:hm=hm.sort_index(ascending=False)
            fig,ax=plt.subplots(figsize=(10,8 if idx=='feature' else 5.5))
            im=ax.imshow(hm.to_numpy(),aspect='auto',cmap='viridis')
            ax.set_xticks(range(len(hm.columns)),hm.columns)
            ax.set_yticks(range(len(hm.index)))
            if idx=='minutes_before_issue':
                ax.set_yticklabels([str(v) if i%3==0 or v==0 else '' for i,v in enumerate(hm.index)])
            else:ax.set_yticklabels(hm.index)
            ax.set(xlabel='Forecast horizon (minutes)',ylabel={'feature':'Predictor','group':'Feature group','minutes_before_issue':'Minutes before issue'}[idx],title=f'{arch}: {title}')
            fig.colorbar(im,ax=ax,label='Mean absolute SHAP (ft)')
            fig.tight_layout();fig.savefig(output/filename,dpi=180);plt.close(fig)
        # Signed, history-summed SHAP explains direction. Color is standardized
        # feature value at ISSUE time only (not a summary of whole 36-step history).
        attr=attribution_by_arch[arch]
        signed=attr['shap_values'].sum(axis=1)
        for h in (60,180):
            hidx=HORIZONS_MIN.index(h)
            tops=ranked_features[ranked_features.architecture==arch].head(15)['feature'].tolist()
            fig,ax=plt.subplots(figsize=(10.0,8.3))
            rng=np.random.default_rng(RANDOM_SEED+1055+hidx)
            scat=None
            for pos,fname in enumerate(tops):
                fidx=FEATURE_NAMES.index(fname)
                jitter=rng.uniform(-0.26,0.26,size=len(signed))
                values=np.clip(attr['issue_feature_scaled'][:,fidx],-3,3)
                scat=ax.scatter(signed[:,fidx,hidx],pos+jitter,c=values,
                                vmin=-3,vmax=3,cmap='coolwarm',s=11,alpha=.55,edgecolors='none')
            ax.set_yticks(range(len(tops)),tops)
            ax.invert_yaxis()
            ax.axvline(0,color='0.4',lw=.8)
            ax.set(xlabel='Signed history-summed SHAP contribution to predicted ΔH (ft)',
                   title=f'{arch}: SHAP summary, {h}-min prediction (composite loss)')
            fig.colorbar(scat,ax=ax,label='Scaled feature value at issue time (clipped to ±3)')
            fig.tight_layout();fig.savefig(output/f'{arch}_shap_beeswarm_{h}min.png',dpi=180);plt.close(fig)
    return ranked_features


def generate_shap_importance(root,datasets,objective=FEATURE_LOSS_NAME,
                             n_samples=SHAP_TEST_SAMPLES,
                             background_samples=SHAP_BACKGROUND_SAMPLES,
                             nsamples=SHAP_NSAMPLES):
    """Explain both fixed composite models, with independent TRAIN background.

    Results are checkpointed per architecture, so incomplete Colab sessions do
    not need to recompute a previously completed, fingerprint-matching SHAP run.
    Existing trained models are read-only. No permutation is performed.
    """
    if objective != FEATURE_LOSS_NAME:
        raise ValueError('SHAP architecture comparison is predeclared for original composite-trained models')
    import shap
    root=Path(root)
    out=root/'shap_feature_importance'
    out.mkdir(parents=True,exist_ok=True)
    frames_f,frames_g,frames_t,attribution_by_arch=[],[],[],{}
    for arch in ('TCN','LSTM'):
        run_dir=root/'loss_experiments'/objective/f'{arch}_fixed2025'
        _,meta=read_run(run_dir)
        if meta.get('loss_name')!=objective:
            raise ValueError(f'Incorrect objective for SHAP model: {arch}')
        checkpoint=run_dir/meta['checkpoint_name']
        if not checkpoint.exists():raise FileNotFoundError(checkpoint)
        expected=_shap_config(arch,objective,checkpoint,datasets,n_samples,background_samples,nsamples)
        npz=out/f'{arch}_shap_attributions.npz'
        meta_path=out/f'{arch}_shap_config.json'
        old_cfg=json.loads(meta_path.read_text()) if meta_path.exists() else None
        if npz.exists() and old_cfg==expected:
            with np.load(npz,allow_pickle=False) as saved:
                attr={name:saved[name] for name in saved.files}
            print(f'[SHAP REUSE] {arch}: validated existing explanations')
        else:
            state=torch.load(checkpoint,map_location=DEVICE,weights_only=True)
            if state.get('loss_name')!=objective:
                raise ValueError(f'Checkpoint loss mismatch for {arch}: {checkpoint}')
            model=make_architecture(arch)
            model.load_state_dict(state['model_state'])
            print(f'[SHAP] Computing expected-gradient attributions for {arch} ({len(HORIZONS_MIN)} horizons)')
            attr=shap_gradient_attributions(model,datasets['train'],datasets['test'],
                                            n_samples=n_samples,background_samples=background_samples,
                                            nsamples=nsamples)
            # Atomic save avoids treating an interrupted write as a complete run.
            tmp=out/f'{arch}_shap_attributions.partial.npz'
            np.savez_compressed(tmp,**attr)
            tmp.replace(npz)
            meta_path.write_text(json.dumps(expected,indent=2)+'\n')
            del model
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
        attribution_by_arch[arch]=attr
        f,g,t=shap_tables(arch,attr)
        frames_f.append(f);frames_g.append(g);frames_t.append(t)
        print(f'[SHAP] {arch}: expected-gradient additivity MAE (ft) per horizon:',
              np.round(attr['expected_gradient_additivity_mae_ft'],5).tolist())
    ranks=plot_shap_results(pd.concat(frames_f,ignore_index=True),
                            pd.concat(frames_g,ignore_index=True),
                            pd.concat(frames_t,ignore_index=True),
                            attribution_by_arch,out)
    print(f'[SHAP] Saved attribution tables, rankings, summary and temporal plots: {out}')
    return ranks


def run_loss_study(mode="all", cache_dir=None, output_root=None,
                   losses=LOSS_NAMES, feature_importance=True,
                   importance_samples=SHAP_TEST_SAMPLES,
                   shap_background=SHAP_BACKGROUND_SAMPLES,
                   shap_nsamples=SHAP_NSAMPLES,
                   download_missing=True):
    """Runs 8 experiment arms; completed runs are re-used after hash/design checks.

    Modes: 'prepare' (cache/feature audit), 'all' (train/reuse and aggregate),
    'report' (aggregate all completed runs without loading raw cache).
    Run a subset of `losses` to finish Colab jobs in separate sessions. The
    aggregate outputs appear only after all four objectives are complete.
    """
    global CACHE_DIR, ALLOW_DOWNLOAD, ARCHITECTURE_LABEL, OUTPUT_DIR
    if mode not in {"prepare","all","report"}:
        raise ValueError("mode must be 'prepare', 'all', or 'report'")
    losses=tuple(losses)
    if not losses or len(set(losses))!=len(losses) or any(l not in LOSS_NAMES for l in losses):
        raise ValueError("Use unique losses from composite, RMSE, NSE, KGE")
    root=Path(output_root or (ROOT/"crabtree_loss_study_results")).expanduser().resolve()
    root.mkdir(parents=True,exist_ok=True)
    if mode=="report":
        table=collect_study_metrics(root,require_complete=True)
        summary,_=plot_loss_study(table,root/"loss_sensitivity")
        return summary
    ALLOW_DOWNLOAD=bool(download_missing)
    CACHE_DIR=Path(cache_dir or (ROOT/"shared_usgs_cache")).expanduser().resolve()
    ensure_cache(CACHE_DIR, download_missing=download_missing)
    ARCHITECTURE_LABEL="TCN"
    OUTPUT_DIR=root/"loss_experiments"/"composite"/"TCN_fixed2025"
    print(f"DEVICE={DEVICE} BATCH_SIZE={BATCH_SIZE} MC_PASSES={MC_DROPOUT_PASSES}")
    datasets,scaler,scale,base=prepare_data()
    ref=training_reference_from_dataset(datasets["train"])
    if mode=="prepare":
        audit={"data":base,"train_loss_reference":ref,"study_losses":list(losses)}
        path=root/"data_audit.json";path.write_text(json.dumps(audit,indent=2)+"\n")
        print(f"Data audit: {path}")
        return audit
    for loss in losses:
        for arch in ("TCN","LSTM"):
            run_architecture_loss(arch,loss,datasets,scaler,scale,base,ref,root)
        # Restore original paired-manifest validation and plots for each objective.
        run_comparison(root/"loss_experiments"/loss)
    table=collect_study_metrics(root,require_complete=False)
    if table is None:
        print("[PARTIAL] Some losses remain. Run the missing loss(es), then mode='report'.")
    else:
        plot_loss_study(table,root/"loss_sensitivity")
        print("[COMPLETE] Consolidated loss sensitivity tables and horizon plots.")
    if feature_importance:
        comp_path=root/"loss_experiments"/FEATURE_LOSS_NAME
        if all((comp_path/f"{a}_fixed2025"/"run_manifest.json").exists() for a in ("TCN","LSTM")):
            generate_shap_importance(root,datasets,FEATURE_LOSS_NAME,
                                     n_samples=importance_samples,
                                     background_samples=shap_background,
                                     nsamples=shap_nsamples)
        else:
            print("[FEATURES] Composite models not completed; feature importance deferred.")
    return table


## 3. Execute the selected experiments

By default: 4 objectives × 2 architectures = **8 distinct trained models**. Every completed arm is checkpointed on Google Drive. The 2025 predictions for each are made with 100 MC-dropout passes. Paired-run manifests and issue times are checked before comparisons.

If you run only one or two losses, the notebook saves those outputs and explicitly reports that the complete sensitivity table must wait for the other objectives. Feature importance for the predefined **composite** models may run once both composite models exist.
SHAP runs for the **composite loss only** after its two models have completed, using a background sampled from 2020–2023. Rankings are expressed as |SHAP| in feet, not decrease in test skill. The `shap_feature_importance/` directory contains reusable per-architecture SHAP attribution archives, CSV ranking tables, time-lag and horizon heatmaps, and signed summary plots at 60 and 180 minutes.


In [ ]:
result = run_loss_study(
    mode=RUN_MODE,
    cache_dir=USER_CACHE_DIR,
    output_root=USER_OUTPUT_ROOT,
    losses=RUN_LOSSES,
    feature_importance=RUN_FEATURE_IMPORTANCE,
    importance_samples=SHAP_EXPLAIN_ISSUES,
    shap_background=SHAP_BACKGROUND_ISSUES,
    shap_nsamples=SHAP_GRADIENT_SAMPLES,
    download_missing=DOWNLOAD_MISSING_CACHE,
)
from IPython.display import display
if isinstance(result, pd.DataFrame):
    if {'architecture', 'training_loss', 'horizon_min'}.issubset(result.columns):
        display(result[['architecture','training_loss','horizon_min',
                        'RMSE_ft','MASE','MDA_raw','NSE_stage','KGE_stage',
                        'skill_vs_persistence']].head(24))
    else:
        display(result)
else:
    print('Preparation finished, or some training objectives remain. Examine the output folder.')

[USGS] Complete cache found; no network requests required.
Validated all 18 annual observation caches in /content/drive/MyDrive/usgs_cache
DEVICE=cuda BATCH_SIZE=512 MC_PASSES=100
  [CACHE] 2020_02087275_00045_hw70_precip.csv: 104,049 rows
  [CACHE] 2021_02087275_00045_hw70_precip.csv: 105,124 rows
  [CACHE] 2022_02087275_00045_hw70_precip.csv: 86,261 rows
  [CACHE] 2023_02087275_00045_hw70_precip.csv: 104,765 rows
  [CACHE] 2024_02087275_00045_hw70_precip.csv: 105,125 rows
  [CACHE] 2025_02087275_00045_hw70_precip.csv: 102,990 rows
  [CACHE] 2020_02087275_00065_hw70_stage.csv: 104,694 rows
  [CACHE] 2021_02087275_00065_hw70_stage.csv: 104,751 rows
  [CACHE] 2022_02087275_00065_hw70_stage.csv: 104,695 rows
  [CACHE] 2023_02087275_00065_hw70_stage.csv: 105,104 rows
  [CACHE] 2024_02087275_00065_hw70_stage.csv: 105,363 rows
  [CACHE] 2025_02087275_00065_hw70_stage.csv: 105,108 rows
  [CACHE] 2020_02087324_00065_us1_stage.csv: 34,947 rows
  [CACHE] 2021_02087324_00065_us1_stage.csv: 35,03

,architecture,training_loss,horizon_min,RMSE_ft,MASE,MDA_raw,NSE_stage,KGE_stage,skill_vs_persistence
0,TCN,composite,15,0.024461,0.830283,0.322704,0.999536,0.998612,0.362058
1,TCN,composite,30,0.051986,1.487777,0.540181,0.997905,0.997448,0.405741
2,TCN,composite,60,0.111088,2.757142,0.687378,0.990436,0.989943,0.433040
3,TCN,composite,90,0.168979,4.077612,0.705205,0.977871,0.978141,0.427492
4,TCN,composite,120,0.223057,5.349846,0.759956,0.961441,0.962778,0.422184
5,TCN,composite,150,0.274718,6.658699,0.798090,0.941513,0.944013,0.409265
6,TCN,composite,180,0.322440,8.014915,0.797127,0.919431,0.925347,0.390949
7,LSTM,composite,15,0.032687,0.903483,0.368918,0.999172,0.997847,0.305815
8,LSTM,composite,30,0.064871,1.618596,0.512300,0.996738,0.994234,0.353488
9,LSTM,composite,60,0.129251,3.021937,0.726266,0.987053,0.985087,0.378589


In [ ]:
print(f"Result DataFrame rows: {len(result)} of 56 expected.")
if len(result) == 56:
    print("SUCCESS: The script fully executed all 8 experiment arms (4 loss functions x 2 architectures) and saved the comparative results!")
else:
    print("PARTIAL: The script did not complete all configurations.")

Result DataFrame rows: 56 of 56 expected.
SUCCESS: The script fully executed all 8 experiment arms (4 loss functions x 2 architectures) and saved the comparative results!


## 4. Where to find the scientific results

```text
crabtree_loss_study_results/
├── loss_experiments/
│   ├── composite/  ── TCN_fixed2025/, LSTM_fixed2025/, comparison_fixed2025/
│   ├── RMSE/       ── TCN_fixed2025/, LSTM_fixed2025/, comparison_fixed2025/
│   ├── NSE/        ── TCN_fixed2025/, LSTM_fixed2025/, comparison_fixed2025/
│   └── KGE/        ── TCN_fixed2025/, LSTM_fixed2025/, comparison_fixed2025/
├── loss_sensitivity/
│   ├── loss_metrics_by_horizon.csv
│   ├── loss_sensitivity_mean_across_7_horizons.csv
│   ├── loss_sensitivity_changes_vs_composite_by_horizon.csv
│   ├── loss_sensitivity_changes_vs_composite_summary.csv
│   ├── TCN_loss_sensitivity_heatmap.png
│   ├── LSTM_loss_sensitivity_heatmap.png
│   └── [TCN/LSTM]_[RMSE/MASE/MDA/NSE/KGE/skill]_across_horizons.png
└── shap_feature_importance/
    ├── TCN_shap_attributions.npz, LSTM_shap_attributions.npz
    ├── shap_feature_importance_by_horizon.csv
    ├── shap_group_importance_by_horizon.csv
    ├── shap_feature_importance_ranked.csv
    ├── shap_group_importance_ranked.csv
    ├── shap_temporal_importance.csv
    ├── TCN_shap_feature_ranking.png, LSTM_shap_feature_ranking.png
    ├── TCN_shap_group_by_horizon.png, LSTM_shap_group_by_horizon.png
    ├── TCN_shap_temporal_by_horizon.png, LSTM_shap_temporal_by_horizon.png
    └── [other horizon heatmaps and 60/180-min SHAP summary plots]
```

**How to interpret the sensitivity table:** `RMSE_ft_change_pct` and `MASE_change_pct` below zero mean a reduction in error; `MDA_raw_change_pp` and `skill_vs_persistence_change_pp` above zero mean better directional accuracy and persistence skill. `NSE_stage_change_absolute` and `KGE_stage_change_absolute` above zero mean improvement. The heatmaps invert the sign of error changes so **positive values always mean improvement**. Heatmap columns have different units—compare signs and within-column magnitudes, not raw values across columns.

**Important:** This produces analysis outputs, not statistical proof that one loss function is superior. The default study fixes a **single random seed** and does not compute confidence intervals; repeat across seeds or add block bootstrap before making strong significance claims. The 2025 test set must remain untouched for tuning.
**Raw USGS cache:** The folder configured as `USER_CACHE_DIR` stores any newly downloaded annual site/parameter CSVs, so subsequent Colab sessions can reuse them without downloading again. If the downloader fails mid-year, it will not save an incomplete CSV; rerun after resolving the USGS/network error.

**SHAP scale:** Attribution metrics and plots use ΔH (feet), not skill points. The 36-timestep predictor history is summed before ranking, so each bar represents the net feature-history contribution. The SHAP background uses **training years 2020–2023**, while explanations use **2025**, with dropout disabled. The output CSVs can be regenerated from the cached `.npz` files.
